# Reasoning Agent: Natural Language → Logic → Classical Inference → True/False

**Aniruddha Bohra (p25cs0201) · AI Assignment 2026**

**Pipeline:** parser → facts + rules (signed literals / FOL clauses) → classical reasoning engine → True/False.

| Problem shape | Representation | Engine |
|---|---|---|
| RuleTaker-style rule bases | Signed literals, one-variable Horn rules | Indexed, agenda-based **forward chaining** with **unification**, stratified negation-as-failure and the **well-founded semantics** |
| CounterBench counterfactuals | Boolean structural equations (AND / OR / NOT) | **Abduction → intervention do(X) → forward propagation** |
| FOLIO-style first-order text | FOL formulas → CNF (Skolemised) | **Resolution refutation**, best-first given-clause search with **set-of-support** and **unit preference** |

## Two methods for free-form English (FOLIO-style problems without formulas)

| | **Method 1: rule-based** (no training) | **Method 2: trained translator** (ours) |
|---|---|---|
| How | Hand-written sentence patterns → formulas | flan-t5-base we fine-tuned on FOLIO training stories → formulas |
| Needs | Python only; instant; fully offline | `torch` + `transformers` + the model (downloaded once) |
| FOLIO validation (English only) | 64.71% | 71.57% |

### How to run each method

- **Method 1, without the trained model (default, fastest):** keep `ENGLISH_METHOD = "rule"` in section 0, then *Run all*. Needs only Python; works fully offline; under 1 ms per problem.
- **Method 2, with the trained model:** set `ENGLISH_METHOD = "trained"`, then *Run all*. About 1 s per English-only problem. Needs `torch` + `transformers` (preinstalled on Colab); the model is downloaded once from Hugging Face, or set `TRAINED_MODEL = "folio-fol-t5"` to use a local copy offline.
- **Either way, both results are printed:** section 13 reports Method 1 on English-only FOLIO, and section 13b evaluates Method 2 and prints a side-by-side comparison. The setting only decides which method makes the predictions in `submission.csv`.
- **Why Method 1 is the default:** the score counts all questions together and speed is also graded. English-only FOLIO problems are a tiny share of the questions, so Method 2's gain (+6 points on those problems) barely changes the overall accuracy, while it can make the run many times slower.

In both methods the translator only produces formulas; **our resolution prover decides every True/False answer** (no generative model ever produces an answer). With `"trained"`, the notebook falls back to Method 1 automatically if the model is unavailable (no torch/transformers or no internet), so it always runs. RuleTaker, CounterBench and FOLIO-with-formulas never use a translator.

## 0. Choose how to run: Method 1 (rule-based, no training) or Method 2 (trained translator)

In [1]:
# ================= CHOOSE THE ENGLISH-TRANSLATION METHOD =================
# "rule"    : Method 1 - rule-based patterns. No training, instant, fully offline.  (DEFAULT: fastest)
# "trained" : Method 2 - our trained translator; falls back to Method 1 automatically if unavailable.
# This setting decides which method makes the PREDICTIONS (section 16). Section 13b always reports
# both methods when the public data is loaded, so the comparison is visible either way.
ENGLISH_METHOD = "rule"

# Which trained model Method 2 uses: a Hugging Face repo id or a local folder.
#   "flexnotop/folio-fol-t5"  - our flan-t5-base, trained with the code in section 17 (default, 71.57%)
#   "folio-fol-t5"            - the same model as a local folder next to this notebook (works offline)
TRAINED_MODEL = "flexnotop/folio-fol-t5"
print(f"English translation method: {ENGLISH_METHOD!r}" + (f"  (model: {TRAINED_MODEL})" if ENGLISH_METHOD == "trained" else ""))

English translation method: 'rule'


## 1. Configuration and helpers

In [2]:
# ============================================================
# 1. Imports, configuration and shared helpers
# ============================================================
import re, time, heapq, itertools, csv, json, os, random
from collections import defaultdict, deque, Counter

CONFIG = {
    # RuleTaker: closed-world (negation as failure) or open-world (explicit negation)
    "RULETAKER_MODE": "CWA",          # "CWA" or "OWA"
    # theories whose negation forms a cycle: "besteffort" ordering or "wfs" (well-founded semantics)
    "NONSTRATIFIED": "wfs",
    # RuleTaker engine: "fc" (closure once per KB, then lookups) or "bc" (goal-directed proof per question)
    "RT_ENGINE": "fc",
    # CounterBench: value assumed for a root variable that is never mentioned
    "CB_UNKNOWN_ROOT": False,
    # CounterBench conventions per question type: (interventions on endogenous variables, evidence use)
    #   "clamp" | "ignore"   and   "abduce" (observations + factual) | "observed" | "prior"
    "CB_POLICY": {t: ("clamp", "abduce") for t in ["basic", "joint", "nested", "conditional", "backdoor"]},
    # FOLIO resolution limits
    "RES_MAX_STEPS": 100,             # longest FOLIO proof needs 18 steps; 100 changes no prediction
    "RES_MAX_CLAUSE_LEN": 8,
    "RES_STRATEGY": "best_first",     # "best_first" (unit preference) or "bfs"
    # Answer used when nothing could be parsed (chosen from public label counts)
    "DEFAULT_LABEL": False,
}

def to_bool_label(v):
    """Map every dataset's label vocabulary onto True / False (entailed / not entailed)."""
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return bool(v)
    s = str(v).strip().lower()
    return s in {"true", "entailment", "entailed", "yes", "1", "proved", "t"}

def split_sentences(text):
    text = text.replace("\n", " ").strip()
    parts = re.split(r"(?<=[.!?])\s+", text)
    return [p.strip().rstrip(".!?").strip() for p in parts if p.strip().rstrip(".!?").strip()]

## 2. RuleTaker parser

In [3]:
# ============================================================
# 2. RuleTaker parser: English -> signed literals and rules
#    Atom  = (predicate, arg1[, arg2])   e.g. ("attr:young", "bob"), ("rel:like", "cat", "dog")
#    Literal = (atom, positive: bool)
#    Variable = "?x"   (RuleTaker rules use one variable)
# ============================================================
VAR = "?x"
PRON = "?p"            # pronoun placeholder, resolved per rule
VAR_WORDS = {"someone", "something", "anyone", "anything", "everyone", "everything",
             "a person", "a thing", "somebody", "anybody", "everybody"}
PRONOUNS = {"they", "it", "he", "she", "them", "this person", "that person", "this thing"}
GENERIC_NOUNS = {"things", "people", "persons", "ones", "animals", "creatures", "thing",
                 "person", "one", "animal", "creature"}
COPULAS = {"is", "are", "am", "was", "were", "be"}
STOP = {"the", "a", "an", "and", "or", "not", "if", "then", "all", "is", "are", "does", "do"}

NATLANG_REWRITES = [
    (r"\btend to be\b", "are"), (r"\btends to be\b", "is"),
    (r"\bare always\b", "are"), (r"\bis always\b", "is"),
    (r"\bare usually\b", "are"), (r"\bis usually\b", "is"),
    (r"\bseem to be\b", "are"), (r"\bseems to be\b", "is"),
    (r"\bwill be\b", "is"), (r"\bmust be\b", "is"), (r"\bcan be\b", "is"),
    (r"\bisn't\b", "is not"), (r"\baren't\b", "are not"),
    (r"\bdoesn't\b", "does not"), (r"\bdon't\b", "do not"),
    (r"\bas well\b", ""), (r"\balso\b", ""), (r"\btoo\b", ""),
]

# all rewrites merged into one precompiled alternation (applied in a single pass)
_REWRITE_MAP = {re.sub(r"\\b", "", pat).replace("\\'", "'"): rep for pat, rep in NATLANG_REWRITES}
_REWRITE_RE = re.compile(r"\b(" + "|".join(re.escape(k) for k in sorted(_REWRITE_MAP, key=len, reverse=True)) + r")\b")
_SPACES = re.compile(r"\s+")

from functools import lru_cache

@lru_cache(maxsize=200_000)
def norm(s):
    s = s.lower().strip().rstrip(".!?").strip()
    s = _REWRITE_RE.sub(lambda m: _REWRITE_MAP[m.group(1)], s)
    s = s.replace(";", ",")
    return _SPACES.sub(" ", s).strip()

def term_of(tokens):
    """Return a normalised term for a token list, or None if it is not a term."""
    if not tokens:
        return None
    phrase = " ".join(tokens)
    if phrase in VAR_WORDS:
        return VAR
    if phrase in PRONOUNS:
        return PRON
    if tokens[0] == "the" and len(tokens) >= 2 and not any(t in STOP for t in tokens[1:]):
        return "_".join(tokens[1:])
    if len(tokens) == 1 and tokens[0] not in STOP and tokens[0] not in COPULAS:
        return tokens[0]
    return None

IRREGULAR = {"has": "have", "does": "do", "goes": "go", "is": "be", "are": "be"}

def singular(w):
    if len(w) <= 3 or w.endswith(("ss", "us", "is", "ous")):
        return w
    if w.endswith("ies"):
        return w[:-3] + "y"
    if w.endswith(("ches", "shes", "xes", "sses")):
        return w[:-2]
    return w[:-1] if w.endswith("s") else w

def verb_base(v):
    if v in IRREGULAR:
        return IRREGULAR[v]
    if v.endswith("ies"):
        return v[:-3] + "y"
    if v.endswith("es") and v[:-2].endswith(("sh", "ch", "x", "ss")):
        return v[:-2]
    if v.endswith("s") and not v.endswith("ss"):
        return v[:-1]
    return v

def split_list(s):
    """'big, red and round' -> ['big', 'red', 'round']"""
    s = s.replace(", and ", ", ")
    out = []
    for part in s.split(","):
        out.extend(p.strip() for p in part.split(" and "))
    return [p for p in out if p]

def parse_clause(text):
    """Parse one simple clause.  Returns (subject_term, [literals]) or None."""
    toks = text.split()
    for i in range(1, len(toks)):
        subj = term_of(toks[:i])
        if subj is None:
            continue
        rest = toks[i:]
        # copula: X is (not) big [and red]
        if rest[0] in COPULAS and len(rest) >= 2:
            pos = True
            body = rest[1:]
            if body[0] == "not":
                pos, body = False, body[1:]
            if body and body[0] in ("a", "an"):
                body = body[1:]
            if not body:
                return None
            lits = []
            for adj in split_list(" ".join(body)):
                apos = pos
                if adj.startswith("not "):
                    apos, adj = False, adj[4:].strip()
                if not adj or " " in adj and not re.fullmatch(r"[a-z]+( [a-z]+)?", adj):
                    return None
                lits.append((("attr:" + adj.replace(" ", "_"), subj), apos))
            return subj, lits
        # modal: X can fly / X cannot fly / X can not fly
        if rest[0] in ("can", "cannot") and len(rest) >= 2:
            neg = rest[0] == "cannot" or rest[1] == "not"
            act = rest[2:] if rest[1] == "not" else rest[1:]
            if act:
                return subj, [(("attr:can_" + "_".join(act), subj), not neg)]
        # negated relation: X does not like Y
        if rest[0] in ("does", "do") and len(rest) >= 4 and rest[1] == "not":
            obj = term_of(rest[3:])
            if obj is not None:
                return subj, [(("rel:" + rest[2], subj, obj), False)]
            return None
        # positive relation: X likes Y
        if len(rest) >= 2:
            obj = term_of(rest[1:])
            if obj is not None and rest[0] not in STOP:
                return subj, [(("rel:" + verb_base(rest[0]), subj, obj), True)]
        continue
    return None

def parse_conj(text, default_subj=None):
    """Parse 'A and B and not C' where later parts may omit the subject."""
    lits, subj = [], default_subj
    for frag in [f.strip() for f in re.split(r"\band\b|,", text) if f.strip()]:
        r = parse_clause(frag)
        if r is None and subj is not None:
            subj_word = {VAR: "someone", PRON: "it"}.get(subj, subj)
            # re-parse with the inherited subject ('not smart' -> 'X is not smart')
            if frag.startswith(("is ", "are ", "does ", "do ")) or frag.split()[0].endswith("s"):
                r = parse_clause(subj_word + " " + frag)
            if r is None:
                r = parse_clause(subj_word + " is " + frag)
            if r is not None:
                r = (subj, [((a[0], subj) + a[2:], p) for a, p in r[1]])
        if r is None:
            return None
        subj = r[0]
        lits.extend(r[1])
    return lits if lits else None

def resolve_pronouns(premises, heads):
    """'it/they' refer to the rule variable, or to the premise subject in ground rules."""
    has_var = any(VAR in a[1:] for a, _ in premises)
    target = VAR
    if not has_var:
        subjects = [a[1] for a, _ in premises if a[1] not in (VAR, PRON)]
        if subjects:
            target = subjects[0]
    fix = lambda lits: [((a[0],) + tuple(target if t == PRON else t for t in a[1:]), p) for a, p in lits]
    return fix(premises), fix(heads)

class Rule:
    __slots__ = ("premises", "head", "text", "id")
    def __init__(self, premises, head, text, rid):
        # positive premises first so variables get bound before negation checks
        self.premises = sorted(premises, key=lambda l: not l[1])
        self.head, self.text, self.id = head, text, rid
    def __repr__(self):
        return f"Rule({self.text})"

def parse_generic_rule(s):
    """'all big, red things are kind' / 'rough people are not nice' / 'people who are big are red'."""
    m = re.match(r"^(?:all |every |each )?(?:(?:things|people|someone|something|anyone|anything|everyone|everything|a person|a thing|persons|ones) (?:who|that|which) (?:is|are) )(.+?) (?:is|are) (not )?(.+)$", s)
    if m:
        conds, neg, concl = m.group(1), m.group(2), m.group(3)
    else:
        m = re.match(r"^(?:all |every |each )?([a-z ,]+?) (?:things|people|persons|ones|animals|creatures|thing|person|one) (?:is|are) (not )?(.+)$", s)
        if not m:
            return None
        conds, neg, concl = m.group(1), m.group(2), m.group(3)
    return _generic_rule_from(conds, neg, concl)

def parse_class_rule(s):
    m = re.match(r"^(?:all )?([a-z]+(?: [a-z]+)?) are (not )?(.+)$", s)
    if not m or m.group(1) in VAR_WORDS:
        return None
    conds = " ".join(singular(w) for w in m.group(1).split()).replace(" ", "_")
    concl = ", ".join(singular(w) for w in split_list(m.group(3)))
    return _generic_rule_from(conds, m.group(2), concl)

def _generic_rule_from(conds, neg, concl):
    prem = []
    for adj in split_list(conds):
        pos = True
        if adj.startswith("not "):
            pos, adj = False, adj[4:]
        if not re.fullmatch(r"[a-z_]+", adj):
            return None
        prem.append((("attr:" + adj, VAR), pos))
    heads = []
    for adj in split_list(concl):
        pos = neg is None
        if adj.startswith("not "):
            pos, adj = False, adj[4:]
        if adj.startswith(("a ", "an ")):
            adj = adj.split(" ", 1)[1]
        if not re.fullmatch(r"[a-z]+", adj):
            return None
        heads.append((("attr:" + adj, VAR), pos))
    return (prem, heads) if prem and heads else None

def parse_rule_sentence(s):
    m = re.match(r"^if (.+?),? then (.+)$", s) or re.match(r"^if (.+?), (.+)$", s)
    if m:
        prem = parse_conj(m.group(1))
        if prem is None:
            return None
        first_subj = prem[0][0][1]
        heads = parse_conj(m.group(2), default_subj=first_subj)
        if heads is None:
            return None
        return resolve_pronouns(prem, heads)
    return parse_generic_rule(s) or parse_class_rule(s)

def parse_ruletaker(context, question=None):
    """Return (facts, rules, query_literals, unparsed_sentences)."""
    facts, rules, unparsed = [], [], []
    for raw in split_sentences(context):
        s = norm(raw)
        if s.startswith(("if ", "all ", "every ", "each ")) or re.match(r"^[a-z]+(?: [a-z]+)? are ", s) or re.search(r"\b(things|people|persons|ones|someone|something) (who|that|are|is)\b", s):
            r = parse_rule_sentence(s)
            if r is not None:
                prem, heads = r
                for h in heads:            # 'then X is a and b' -> one rule per conclusion
                    rules.append(Rule(prem, h, raw, len(rules)))
                continue
        lits = parse_conj(s)
        if lits is not None and all(VAR not in a[1:] and PRON not in a[1:] for a, _ in lits):
            facts.extend(lits)
            continue
        r = parse_rule_sentence(s)
        if r is not None:
            prem, heads = r
            for h in heads:
                rules.append(Rule(prem, h, raw, len(rules)))
            continue
        unparsed.append(raw)
    query = None
    if question is not None:
        q = parse_conj(norm(question))
        if q is not None and all(VAR not in a[1:] and PRON not in a[1:] for a, _ in q):
            query = q
    return facts, rules, query, unparsed

## 3. Forward chaining (main engine) and backward chaining (baseline)

In [4]:
# ============================================================
# 3. Reasoning engine for rule bases
#    * Unification of atoms with one-variable patterns
#    * Indexed, agenda-based (semi-naive) FORWARD CHAINING
#    * Stratified negation-as-failure (CWA) or explicit negation (OWA)
#    * Justifications recorded for reasoning traces
#    * BACKWARD CHAINING baseline for the experiments section
# ============================================================
def is_var(t):
    return isinstance(t, str) and t.startswith("?")

def unify_atom(pattern, fact, theta):
    """Unify a pattern atom with a ground atom; returns extended substitution or None."""
    if pattern[0] != fact[0] or len(pattern) != len(fact):
        return None
    theta = dict(theta)
    for p, f in zip(pattern[1:], fact[1:]):
        if is_var(p):
            if p in theta:
                if theta[p] != f:
                    return None
            else:
                theta[p] = f
        elif p != f:
            return None
    return theta

def subst(atom, theta):
    return (atom[0],) + tuple(theta.get(t, t) if is_var(t) else t for t in atom[1:])

def is_ground(atom):
    return not any(is_var(t) for t in atom[1:])

def compute_strata(rules):
    """Predicate strata so that a rule using 'not p' runs only after p is complete."""
    preds = set()
    for r in rules:
        preds.add(r.head[0][0])
        for a, _ in r.premises:
            preds.add(a[0])
    stratum = {p: 0 for p in preds}
    for _ in range(len(preds) + 1):
        changed = False
        for r in rules:
            h = r.head[0][0]
            for a, pos in r.premises:
                need = stratum[a[0]] + (0 if pos else 1)
                if stratum[h] < need:
                    stratum[h] = need
                    changed = True
        if not changed:
            return stratum, True
    return stratum, False          # negation cycle: not stratifiable, best effort

class ForwardChainer:
    def __init__(self, facts, rules, mode="CWA"):
        self.mode = mode
        self.rules = rules
        self.pos, self.neg = set(), set()
        self.index = defaultdict(set)              # predicate -> ground atoms (positive)
        self.nindex = defaultdict(set)             # predicate -> ground atoms (negative, OWA)
        self.why = {}                              # literal -> (rule, premise literals)
        self.entities = set()
        self.facts = list(facts)
        self.neg_ref = None                        # fixed set used to judge 'not' (alternating fixpoint)
        self.upper = None                          # possibly-true atoms (well-founded semantics)
        self.stratified = True
        for a, p in facts:
            self.entities.update(a[1:])
            self._add((a, p), None, None)
        for r in rules:
            for a, _ in r.premises + [r.head]:
                self.entities.update(t for t in a[1:] if not is_var(t))
        self.inferences = 0

    def _add(self, lit, rule, prem):
        a, p = lit
        target, idx = (self.pos, self.index) if p else (self.neg, self.nindex)
        if a in target:
            return False
        target.add(a)
        idx[a[0]].add(a)
        if rule is not None:
            self.why[lit] = (rule, prem)
        return True

    def holds_neg(self, atom):
        """Truth of 'not atom' at evaluation time."""
        if self.neg_ref is not None:
            return atom not in self.neg_ref        # judged against a fixed reference model
        if self.mode == "CWA":
            return atom not in self.pos            # negation as failure
        return atom in self.neg                    # explicit negation only

    def _match(self, premises, i, theta, used):
        """Backtracking join over the remaining premises; yields (theta, used_literals)."""
        if i == len(premises):
            yield theta, used
            return
        a, pos = premises[i]
        a = subst(a, theta)
        if pos:
            if is_ground(a):
                if a in self.pos:
                    yield from self._match(premises, i + 1, theta, used + [(a, True)])
            else:
                for f in list(self.index.get(a[0], ())):
                    t2 = unify_atom(a, f, theta)
                    if t2 is not None:
                        yield from self._match(premises, i + 1, t2, used + [(f, True)])
        else:
            if is_ground(a):
                if self.holds_neg(a):
                    yield from self._match(premises, i + 1, theta, used + [(a, False)])
            else:                                  # unbound variable under negation: range over entities
                for e in self.entities:
                    t2 = dict(theta); t2[VAR] = e
                    g = subst(a, t2)
                    if self.holds_neg(g):
                        yield from self._match(premises, i + 1, t2, used + [(g, False)])

    def _fire(self, rule, theta_seed, agenda):
        for theta, used in self._match(rule.premises, 0, theta_seed, []):
            head_atom = subst(rule.head[0], theta)
            if not is_ground(head_atom):           # variable only in head: apply to all entities
                for e in self.entities:
                    t2 = dict(theta); t2[VAR] = e
                    self._derive((subst(rule.head[0], t2), rule.head[1]), rule, used, agenda)
            else:
                self._derive((head_atom, rule.head[1]), rule, used, agenda)

    def _derive(self, lit, rule, used, agenda):
        self.inferences += 1
        if self._add(lit, rule, used):
            agenda.append(lit)

    def _run(self, rules):
        """Semi-naive agenda loop: a new fact only wakes rules that mention its predicate."""
        trigger = defaultdict(list)
        for r in rules:
            for k, (a, pos) in enumerate(r.premises):
                if pos or self.mode == "OWA":
                    trigger[(a[0], pos)].append((r, k))
        agenda = deque()
        for r in rules:                            # initial full pass
            self._fire(r, {}, agenda)
        while agenda:
            a, p = agenda.popleft()
            for r, k in trigger.get((a[0], p), ()):
                theta = unify_atom(r.premises[k][0], a, {})
                if theta is not None:
                    self._fire(r, theta, agenda)

    def saturate(self):
        if self.mode == "OWA":
            self._run(self.rules)
            return self
        strata, ok = compute_strata(self.rules)
        self.stratified = ok
        if not ok and CONFIG.get("NONSTRATIFIED") == "wfs":
            return self._well_founded()
        by_s = defaultdict(list)
        for r in self.rules:
            by_s[strata[r.head[0][0]]].append(r)
        for s in sorted(by_s):
            self._run(by_s[s])
        return self

    def _gamma(self, ref):
        """Least model when every 'not a' is decided by the fixed set ref (monotone, plain FC)."""
        kb = ForwardChainer(self.facts, self.rules, "CWA")
        kb.entities = self.entities
        kb.neg_ref = ref
        kb._run(self.rules)
        return kb

    def _well_founded(self):
        """Alternating fixpoint (Van Gelder): T grows from below, U shrinks from above.
        T = surely true, U = possibly true; atoms in U - T are 'undefined'."""
        T = self._gamma(set())                     # placeholder; replaced on first loop
        true_set = set()
        for _ in range(50):
            U = self._gamma(true_set)              # overestimate: negation judged against T
            T = self._gamma(U.pos)                 # underestimate: negation judged against U
            if T.pos == true_set:
                break
            true_set = set(T.pos)
        self.pos, self.index, self.why = T.pos, T.index, T.why
        self.upper = U.pos
        self.inferences = T.inferences + U.inferences
        return self

    def query(self, lit):
        a, p = lit
        if p:
            return a in self.pos
        if self.upper is not None:                 # well-founded: 'not a' needs a to be false, not undefined
            return a not in self.upper
        return self.holds_neg(a)

    def explain(self, lit, depth=0, seen=None):
        """Human-readable proof tree for a derived literal."""
        seen = seen or set()
        a, p = lit
        txt = ("" if p else "NOT ") + f"{a[0].split(':')[1]}({', '.join(a[1:])})"
        pad = "    " * depth
        if lit in seen:
            return [pad + txt + "  (see above)"]
        seen.add(lit)
        if lit in self.why:
            rule, prem = self.why[lit]
            lines = [pad + txt + f"   <= rule: \"{rule.text}\""]
            for q in prem:
                lines += self.explain(q, depth + 1, seen)
            return lines
        if p and a in self.pos:
            return [pad + txt + "   (given fact)"]
        if not p:
            return [pad + txt + ("   (negation as failure: not derivable)" if self.mode == "CWA" else "   (given)")]
        return [pad + txt + "   (not derivable)"]

class BackwardChainer:
    """Goal-directed baseline (SLD-style with path loop check). Used only for comparison."""
    def __init__(self, facts, rules, mode="CWA"):
        self.facts = {a for a, p in facts if p}
        self.negfacts = {a for a, p in facts if not p}
        self.mode = mode
        self.by_head = defaultdict(list)
        for r in rules:
            self.by_head[(r.head[0][0], r.head[1])].append(r)
        self.entities = set()
        for a, _ in facts:
            self.entities.update(a[1:])
        self.memo = {}
        self.inferences = 0

    stratified = True

    def query(self, lit):
        return self.prove(lit[0], lit[1])

    def prove(self, atom, positive=True, path=frozenset()):
        key = (atom, positive)
        if key in self.memo:
            return self.memo[key]
        if self.mode == "CWA" and not positive:
            return not self.prove(atom, True, path)
        if (positive and atom in self.facts) or (not positive and atom in self.negfacts):
            self.memo[key] = True
            return True
        if key in path:
            return False
        path = path | {key}
        for r in self.by_head.get((atom[0], positive), ()):
            theta = unify_atom(r.head[0], atom, {})
            if theta is None:
                continue
            self.inferences += 1
            if self._prove_all(r.premises, 0, theta, path):
                self.memo[key] = True
                return True
        return False                                # failures are not cached (avoids loop-check unsoundness)

    def _prove_all(self, prem, i, theta, path):
        if i == len(prem):
            return True
        a, pos = prem[i]
        a = subst(a, theta)
        cands = [theta] if is_ground(a) else [dict(theta, **{VAR: e}) for e in self.entities]
        for t in cands:
            if self.prove(subst(a, t), pos, path) and self._prove_all(prem, i + 1, t, path):
                return True
        return False

def solve_ruletaker(context, question, mode=None, engine="fc", cache=None, return_kb=False):
    mode = mode or CONFIG["RULETAKER_MODE"]
    facts, rules, query, unparsed = parse_ruletaker(context, question)
    if query is None:
        return (CONFIG["DEFAULT_LABEL"], None) if return_kb else CONFIG["DEFAULT_LABEL"]
    if engine == "bc":
        bc = BackwardChainer(facts, rules, mode)
        ans = all(bc.prove(a, p) for a, p in query)
        return (ans, bc) if return_kb else ans
    kb = cache.get(context) if cache is not None else None
    if kb is None:
        kb = ForwardChainer(facts, rules, mode).saturate()
        if cache is not None:
            cache[context] = kb
    ans = all(kb.query(l) for l in query)
    return (ans, kb) if return_kb else ans

## 4. CounterBench: structural causal model

In [5]:
# ============================================================
# 4. CounterBench: deterministic structural causal model (SCM)
#    Parse "A and B together cause C", "not D causes E", "F or G causes H",
#    "X causes not Y", "X causes A, B and C" into Boolean structural equations,
#    then answer counterfactuals with Pearl's three steps:
#      (1) abduction  - fix root variables consistent with the observations
#      (2) action     - do(X = v): replace X's equation by the constant v
#      (3) prediction - evaluate the outcome in topological order
# ============================================================
def cb_lit(s):
    s = s.strip().lower()
    s = re.sub(r"^(the |that )", "", s)
    neg = False
    while s.startswith("not "):
        neg, s = not neg, s[4:].strip()
    return s.strip(), neg

def cb_parse_expr(s):
    """'not Frim and Trune' -> ('and', [('var','frim',True), ('var','trune',False)])"""
    s = re.sub(r"\b(together|both|jointly)\b", "", s.lower()).strip()
    disj = []
    for d in re.split(r"\bor\b", s):
        conj = [cb_lit(c) for c in re.split(r"\band\b|,", d) if c.strip()]
        disj.append(("and", [("var", v, n) for v, n in conj]))
    return ("or", disj)

def cb_eval(expr, val):
    if expr[0] == "not":
        return not cb_eval(expr[1], val)
    if expr[0] == "or":
        return any(cb_eval(e, val) for e in expr[1])
    if expr[0] == "and":
        return all(cb_eval(e, val) for e in expr[1])
    _, v, neg = expr
    return (not val[v]) if neg else val[v]

def cb_vars(expr):
    if expr[0] == "not":
        return cb_vars(expr[1])
    if expr[0] in ("or", "and"):
        return set().union(*[cb_vars(e) for e in expr[1]]) if expr[1] else set()
    return {expr[1]}

def parse_counterbench(text):
    t = re.sub(r"\s+", " ", text.replace("\n", " ")).strip()
    low = t.lower()
    # ---- causal relations: prefer the 'We know that ...' sentence
    m = re.search(r"we know that (.+?)\.(?: |$)", low)
    if m:
        rel_text, verb = m.group(1), r"\bcauses?\b"
    else:
        m = re.search(r":\s*(.+?)\.(?: |$)", low)
        rel_text, verb = (m.group(1) if m else low), r"\bha(?:s|ve) (?:a )?direct effects? on\b"
    rel_text = rel_text.replace(", and ", ", ")
    clauses = []
    for frag in [f.strip() for f in rel_text.split(",") if f.strip()]:
        if re.search(verb, frag) or not clauses:
            clauses.append(frag)
        else:                                   # continuation of an effect list
            clauses[-1] += ", " + frag
    eqs = {}
    for c in clauses:
        parts = re.split(verb, c, maxsplit=1)
        if len(parts) != 2:
            continue
        cause, effects = cb_parse_expr(parts[0]), parts[1]
        for eff in [e for e in re.split(r",|\band\b", effects) if e.strip()]:
            v, neg = cb_lit(eff)
            e = ("not", cause) if neg else cause
            eqs[v] = ("or", [eqs[v], e]) if v in eqs else e     # several causes -> OR
    # ---- query
    qm = re.search(r"would (.+?) (?:occur|happen|be true)", low)
    target = cb_lit(qm.group(1))[0] if qm else None
    allvars = set(eqs) | set().union(*[cb_vars(e) for e in eqs.values()]) if eqs else set()
    names = sorted(allvars, key=len, reverse=True)
    pat = re.compile(r"((?:not )*)\b(" + "|".join(re.escape(n) for n in names) + r")\b") if names else None

    def lits(s):
        return [(mm.group(2), mm.group(1).count("not ") % 2 == 0) for mm in pat.finditer(s)] if pat else []

    observed = []
    for om in re.finditer(r"we (?:observed|observe|see|saw|noticed) (.+?)(?:\.|\?|\bwould\b|\bassume\b|\bsuppose\b|$)", low):
        if "cause" not in om.group(1):
            observed += lits(om.group(1))
    hyp, factual = [], []
    for am in re.finditer(r"(?:assume|suppose|imagine) (?!a self)(.+?)(?:\.|\?|$)", low):
        hyp += lits(re.sub(r"based on this assumption|further", "", am.group(1)))
    im = re.search(r"would .+? (?:occur|happen|be true) if (.+?)(?:\?|$)", low)
    if im:
        h = im.group(1)
        if " instead of " in h:
            h, f = h.split(" instead of ", 1)
            factual += lits(f)
        hyp += [l for l in lits(h)]
    return {"eqs": eqs, "target": target, "do": hyp, "observed": observed, "factual": factual, "vars": allvars}

def cb_eval3(expr, val):
    """Kleene three-valued evaluation: True / False / None (unknown)."""
    op = expr[0]
    if op == "not":
        r = cb_eval3(expr[1], val)
        return None if r is None else not r
    if op in ("or", "and"):
        rs = [cb_eval3(e, val) for e in expr[1]]
        if op == "or":
            return True if True in rs else (False if all(r is False for r in rs) else None)
        return False if False in rs else (True if all(r is True for r in rs) else None)
    _, v, neg = expr
    x = val.get(v)
    return None if x is None else ((not x) if neg else x)

def scm_eval(eqs, roots_val, do, allvars=()):
    """Propagate values to a fixpoint; works on cyclic graphs thanks to short-circuit logic."""
    val = dict(roots_val)
    val.update(do)
    pending = [v for v in eqs if v not in val]
    while pending:
        changed = False
        for v in list(pending):
            r = cb_eval3(eqs[v], val)
            if r is not None:
                val[v] = r
                pending.remove(v)
                changed = True
        if not changed:                         # genuine cycle with no forced value: break it with the default
            val[pending.pop(0)] = CONFIG["CB_UNKNOWN_ROOT"]
    for v in allvars:
        val.setdefault(v, CONFIG["CB_UNKNOWN_ROOT"])
    return val

def cb_qtype(text):
    t = text.lower()
    if "bern(" in t:
        return "backdoor"
    if re.search(r"\b(assume|suppose)\b", t):
        return "nested"
    if re.search(r"\bwe (observed|observe)\b", t):
        return "conditional"
    m = re.search(r"would .+? (?:occur|happen) if (.+?)(?:\?|$)", t)
    if m and re.search(r"\band\b", m.group(1).split(" instead of ")[0]):
        return "joint"
    return "basic"

def solve_counterbench(text, return_info=False, policy=None):
    q = parse_counterbench(text)
    eqs, target = q["eqs"], q["target"]
    if not eqs or target is None:
        return CONFIG["DEFAULT_LABEL"]
    qtype = cb_qtype(text)
    endo_do, evid = policy or CONFIG["CB_POLICY"].get(qtype, ("clamp", "abduce"))
    do_items = q["do"]
    if endo_do == "ignore":                     # convention: drop interventions on endogenous variables
        kept = [(v, b) for v, b in do_items if v not in eqs]
        do_items = kept or do_items
    do = dict(do_items)
    bern = {m.group(1): float(m.group(2)) for m in re.finditer(r"(\w+)\s*~\s*bern\(\s*([\d.]+)\s*\)", text.lower())}
    roots = sorted(v for v in q["vars"] if v not in eqs)
    if evid == "abduce":
        evidence = dict(q["observed"] + q["factual"])
    elif evid == "observed":
        evidence = dict(q["observed"])
    else:
        evidence = {}
    fixed = {r: evidence[r] for r in roots if r in evidence}
    free = [r for r in roots if r not in fixed and r not in do]
    # (1) abduction over free roots, weighted by Bernoulli priors where given
    w_true = w_false = 0.0
    outcomes = []
    for bits in itertools.product([True, False], repeat=min(len(free), 12)):
        rv = dict(fixed); rv.update(zip(free, bits))
        world = scm_eval(eqs, rv, {}, q["vars"])
        if all(world.get(k) == v for k, v in evidence.items()):
            y = scm_eval(eqs, rv, do, q["vars"]).get(target, CONFIG["CB_UNKNOWN_ROOT"])   # (2) do + (3) predict
            w = 1.0
            for r, b in zip(free, bits):
                p = bern.get(r, 0.5)
                w *= p if b else 1 - p
            outcomes.append(y)
            if y: w_true += w
            else: w_false += w
    if outcomes and len(set(outcomes)) == 1:
        ans = outcomes[0]
    elif bern and abs(w_true - w_false) > 1e-9:
        ans = w_true > w_false
    else:
        rv = dict(fixed)
        rv.update({r: (bern[r] > 0.5 if r in bern else CONFIG["CB_UNKNOWN_ROOT"]) for r in free})
        ans = scm_eval(eqs, rv, do, q["vars"]).get(target, CONFIG["DEFAULT_LABEL"])
    if return_info:
        return ans, q
    return ans

## 5. First-order logic and resolution

In [6]:
# ============================================================
# 5. First-order logic: parser -> CNF (with Skolemisation) -> RESOLUTION REFUTATION
#    Search strategy: best-first given-clause loop with set-of-support and unit preference
#    KB |= Q   iff   KB ∧ ¬Q derives the empty clause
# ============================================================
class FOLError(Exception):
    pass

FOL_TOKEN = re.compile(r"\s*(∀|∃|¬|∧|∨|→|↔|⟷|⊕|<->|->|&|\||~|!|\(|\)|,|=|≠|[\w'’.\-]+)")
SYM = {"<->": "↔", "⟷": "↔", "->": "→", "&": "∧", "|": "∨", "~": "¬", "!": "¬"}

def balance_brackets(s):
    """Annotations occasionally have a stray ')' or a missing one."""
    depth, out = 0, []
    for ch in s:
        if ch == "(":
            depth += 1
        elif ch == ")":
            if depth == 0:
                continue
            depth -= 1
        out.append(ch)
    return "".join(out) + ")" * depth

def fol_tokens(s):
    s = s.strip()
    out, i = [], 0
    while i < len(s):
        m = FOL_TOKEN.match(s, i)
        if not m:
            if s[i].isspace():
                i += 1; continue
            raise FOLError(f"bad char {s[i]!r}")
        tok = SYM.get(m.group(1), m.group(1))
        out.append(tok)
        i = m.end()
    return out

class FOLParser:
    def __init__(self, text):
        self.t, self.i, self.bound = fol_tokens(text), 0, []
    def peek(self):
        return self.t[self.i] if self.i < len(self.t) else None
    def eat(self, tok=None):
        cur = self.peek()
        if tok is not None and cur != tok:
            raise FOLError(f"expected {tok}, got {cur}")
        self.i += 1
        return cur
    def parse(self):
        f = self.iff()
        if self.peek() is not None:
            raise FOLError(f"trailing token {self.peek()}")
        return f
    def iff(self):
        a = self.imp()
        while self.peek() == "↔":
            self.eat(); a = ("iff", a, self.imp())
        return a
    def imp(self):
        a = self.orx()
        if self.peek() == "→":
            self.eat(); return ("imp", a, self.imp())
        return a
    def orx(self):
        a = self.andx()
        while self.peek() in ("∨", "⊕"):
            op = "or" if self.eat() == "∨" else "xor"
            a = (op, a, self.andx())
        return a
    def andx(self):
        a = self.unary()
        while self.peek() == "∧":
            self.eat(); a = ("and", a, self.unary())
        return a
    def unary(self):
        tok = self.peek()
        if tok == "¬":
            self.eat(); return ("not", self.unary())
        if tok in ("∀", "∃"):
            self.eat()
            v = self.eat()
            self.bound.append(v)
            body = self.unary() if self.peek() in ("∀", "∃") else self.iff()
            self.bound.pop()
            return ("all" if tok == "∀" else "ex", v, body)
        if tok == "(":
            self.eat(); f = self.iff(); self.eat(")"); return f
        return self.atom()
    def term(self):
        name = self.eat()
        if self.peek() == "(":
            self.eat(); args = [self.term()]
            while self.peek() == ",":
                self.eat(); args.append(self.term())
            self.eat(")")
            return ("fn", name, tuple(args))
        return ("var", name) if name in self.bound else ("const", name.lower())
    def atom(self):
        name = self.eat()
        if name is None or not re.match(r"\w", name):
            raise FOLError(f"unexpected {name}")
        args = ()
        if self.peek() == "(":
            self.eat(); a = [self.term()]
            while self.peek() == ",":
                self.eat(); a.append(self.term())
            self.eat(")"); args = tuple(a)
        if self.peek() in ("=", "≠"):            # infix equality, treated as a predicate
            op = self.eat(); rhs = self.term()
            lhs = ("var", name) if name in self.bound else ("const", name.lower())
            at = ("pred", "Eq", (lhs, rhs))
            return at if op == "=" else ("not", at)
        return ("pred", name, args)

def fol_parse(s):
    try:
        return FOLParser(s).parse()
    except FOLError:
        return FOLParser(balance_brackets(s)).parse()

# ---- CNF conversion ----
def elim(f):
    op = f[0]
    if op == "pred": return f
    if op == "not": return ("not", elim(f[1]))
    if op in ("all", "ex"): return (op, f[1], elim(f[2]))
    a, b = elim(f[1]), elim(f[2])
    if op == "imp": return ("or", ("not", a), b)
    if op == "iff": return ("and", ("or", ("not", a), b), ("or", a, ("not", b)))
    if op == "xor": return ("and", ("or", a, b), ("or", ("not", a), ("not", b)))
    return (op, a, b)

def nnf(f, neg=False):
    op = f[0]
    if op == "pred": return ("not", f) if neg else f
    if op == "not": return nnf(f[1], not neg)
    if op in ("all", "ex"):
        q = op if not neg else ("ex" if op == "all" else "all")
        return (q, f[1], nnf(f[2], neg))
    if op in ("and", "or"):
        o = op if not neg else ("or" if op == "and" else "and")
        return (o, nnf(f[1], neg), nnf(f[2], neg))
    raise FOLError(op)

_fresh = itertools.count()
def term_sub(t, env):
    if t[0] == "var": return env.get(t[1], t)
    if t[0] == "fn": return ("fn", t[1], tuple(term_sub(a, env) for a in t[2]))
    return t

def skolemize(f, univ=(), env=None):
    env = env or {}
    op = f[0]
    if op == "pred": return ("pred", f[1], tuple(term_sub(a, env) for a in f[2]))
    if op == "not": return ("not", skolemize(f[1], univ, env))
    if op == "all":
        nv = ("var", f"V{next(_fresh)}")
        return skolemize(f[2], univ + (nv,), dict(env, **{f[1]: nv}))
    if op == "ex":
        sk = ("fn", f"sk{next(_fresh)}", tuple(univ)) if univ else ("const", f"sk{next(_fresh)}")
        return skolemize(f[2], univ, dict(env, **{f[1]: sk}))
    return (op, skolemize(f[1], univ, env), skolemize(f[2], univ, env))

def to_t(t):
    if t[0] == "var": return "?" + t[1]
    if t[0] == "const": return t[1]
    return (t[1],) + tuple(to_t(a) for a in t[2])

def clauses_of(f, limit=256):
    op = f[0]
    if op == "pred": return [frozenset([(True, f[1], tuple(to_t(a) for a in f[2]))])]
    if op == "not":
        a = f[1]; return [frozenset([(False, a[1], tuple(to_t(x) for x in a[2]))])]
    if op == "and": return clauses_of(f[1], limit) + clauses_of(f[2], limit)
    if op == "or":
        A, B = clauses_of(f[1], limit), clauses_of(f[2], limit)
        if len(A) * len(B) > limit: raise FOLError("CNF blow-up")
        return [a | b for a in A for b in B]
    raise FOLError(op)

def _drop_top(f):
    """'_top(x)' means 'true' (an unrestricted universal); simplify it away."""
    op = f[0]
    if op == "pred":
        return ("true",) if f[1] == "_top" else f
    if op == "not":
        a = _drop_top(f[1]); return ("false",) if a == ("true",) else ("not", a)
    if op in ("all", "ex"):
        return (op, f[1], _drop_top(f[2]))
    a, b = _drop_top(f[1]), _drop_top(f[2])
    if op == "and":
        return b if a == ("true",) else a if b == ("true",) else (op, a, b)
    if op == "imp":
        return b if a == ("true",) else (op, a, b)
    if op == "or":
        return ("true",) if ("true",) in (a, b) else (op, a, b)
    return (op, a, b)

def to_cnf(formula):
    formula = _drop_top(formula)
    if formula == ("true",):
        return []
    cl = clauses_of(skolemize(nnf(elim(formula))))
    return [c for c in cl if not any((not s, p, a) in c for s, p, a in c)]   # drop tautologies

# ---- unification over terms (strings / tuples) ----
def t_is_var(t): return isinstance(t, str) and t.startswith("?")

def walk(t, th):
    while t_is_var(t) and t in th: t = th[t]
    return t

def occurs(v, t, th):
    t = walk(t, th)
    if t == v: return True
    return isinstance(t, tuple) and any(occurs(v, a, th) for a in t[1:])

def unify(x, y, th):
    if th is None: return None
    x, y = walk(x, th), walk(y, th)
    if x == y: return th
    if t_is_var(x): return None if occurs(x, y, th) else {**th, x: y}
    if t_is_var(y): return None if occurs(y, x, th) else {**th, y: x}
    if isinstance(x, tuple) and isinstance(y, tuple) and len(x) == len(y) and x[0] == y[0]:
        for a, b in zip(x[1:], y[1:]):
            th = unify(a, b, th)
            if th is None: return None
        return th
    return None

def resolve_t(t, th):
    t = walk(t, th)
    return (t[0],) + tuple(resolve_t(a, th) for a in t[1:]) if isinstance(t, tuple) else t

def lit_sub(l, th): return (l[0], l[1], tuple(resolve_t(a, th) for a in l[2]))

def rename(clause, tag):
    def r(t):
        if t_is_var(t): return t + tag
        return (t[0],) + tuple(r(a) for a in t[1:]) if isinstance(t, tuple) else t
    return frozenset((s, p, tuple(r(a) for a in args)) for s, p, args in clause)

def canon(clause):
    names = {}
    def r(t):
        if t_is_var(t): return names.setdefault(t, f"?{len(names)}")
        return (t[0],) + tuple(r(a) for a in t[1:]) if isinstance(t, tuple) else t
    lits = sorted(clause, key=repr)
    return tuple(sorted(((s, p, tuple(r(a) for a in args)) for s, p, args in lits), key=repr))

def depth_of(t):
    return 1 + max((depth_of(a) for a in t[1:]), default=0) if isinstance(t, tuple) else 0

def resolvents(c1, c2):
    c2 = rename(c2, "'")
    for l1 in c1:
        for l2 in c2:
            if l1[0] != l2[0] and l1[1] == l2[1] and len(l1[2]) == len(l2[2]):
                th = {}
                for a, b in zip(l1[2], l2[2]):
                    th = unify(a, b, th)
                    if th is None: break
                if th is None: continue
                new = frozenset(lit_sub(l, th) for l in (c1 - {l1}) | (c2 - {l2}))
                if any((not s, p, a) in new for s, p, a in new):
                    continue                                  # tautology
                yield new

def resolution_prove(kb_clauses, goal_clauses, max_steps=None, strategy=None, trace=None):
    """Given-clause loop; returns (proved, steps)."""
    max_steps = max_steps or CONFIG["RES_MAX_STEPS"]
    strategy = strategy or CONFIG["RES_STRATEGY"]
    maxlen = CONFIG["RES_MAX_CLAUSE_LEN"]
    usable = list(kb_clauses)
    seen = {canon(c) for c in usable}
    heap, counter = [], itertools.count()
    parents = {}
    def push(c, depth, par=None):
        k = canon(c)
        if k in seen or len(c) > maxlen: return
        if any(depth_of(a) > 3 for _, _, args in c for a in args): return
        seen.add(k)
        c = frozenset(k)                       # store in canonical variable names
        parents[c] = par
        prio = (len(c), depth) if strategy == "best_first" else (depth,)
        heapq.heappush(heap, (prio, next(counter), depth, c))
    for g in goal_clauses:
        if not g: return True, 0
        push(g, 0)
    steps = 0
    while heap and steps < max_steps:
        _, _, d, given = heapq.heappop(heap)
        steps += 1
        for other in usable + [given]:
            for r in resolvents(given, other):
                if not r:
                    if trace is not None:
                        trace.extend([given, other])
                    return True, steps
                push(r, d + 1, (given, other))
        usable.append(given)
    return False, steps

def fol_entails(premise_formulas, conclusion_formula, **kw):
    kb = []
    for f in premise_formulas:
        kb.extend(to_cnf(f))
    goal = to_cnf(("not", conclusion_formula))
    return resolution_prove(kb, goal, **kw)

## 6. Method 1: rule-based English → FOL translator (FOLIO-style text)

In [7]:
# ============================================================
# 6. FOLIO-style English -> FOL (rule-based, no LLM, runs offline)
#    Covers common shapes: All/Every/Each X are Y, No X are Y, Some X are Y,
#    If ... then ..., <Name> is (not) (either) A (or B), <Name> verbs ...
#    Sentences that do not match are skipped; the prover only uses what parsed,
#    so a parse failure can only make the answer fall back to "not entailed".
# ============================================================
QUANT_HEADS = {"all", "every", "each", "any", "no", "some", "if", "there", "a", "an", "the",
               "anyone", "everyone", "someone", "somebody", "nobody", "people", "everything",
               "something", "anything", "none", "many", "most"}
DROP = {"a", "an", "the", "is", "are", "be", "been", "being", "who", "that", "which", "does", "do"}
GENERIC = {"people", "person", "persons", "thing", "things", "one", "ones", "someone", "something",
           "everyone", "everything", "anyone", "anything", "individual", "individuals"}

def stem(w):
    if len(w) <= 3 or w.endswith(("ss", "us", "is", "ous")):
        return w
    return verb_base(w)

def nl_pred(phrase):
    words = [stem(w) for w in re.findall(r"[a-z0-9]+", phrase.lower()) if w not in DROP]
    return "_".join(words) if words else None

def nl_atom(phrase, term):
    p = nl_pred(phrase)
    if not p:
        raise FOLError("empty predicate")
    return ("pred", p, (term,))

def nl_body(phrase, term):
    """Predicate phrase about one term, with not / and / or / either-or / neither-nor."""
    ph = re.sub(r"^(?:she|he|they|it) (?:is |are |does |do )?", "", phrase.strip())
    ph = re.sub(r"^(?:is|are) (she|he|they|it) ", "", ph)
    m = re.match(r"^neither (.+?) nor (.+)$", ph)
    if m:
        return ("and", ("not", nl_body(m.group(1), term)), ("not", nl_body(m.group(2), term)))
    m = re.match(r"^(?:is |are )?either (.+?) or (.+)$", ph)
    if m:
        return ("xor", nl_body(m.group(1), term), nl_body(m.group(2), term))
    if " or " in ph:
        a, b = ph.split(" or ", 1)
        return ("or", nl_body(a, term), nl_body(b, term))
    if " and " in ph:
        a, b = ph.split(" and ", 1)
        return ("and", nl_body(a, term), nl_body(b, term))
    m = re.match(r"^(?:is not|are not|was not|were not|isn't|aren't|wasn't|weren't|does not|do not|did not|doesn't|don't|didn't|"
                 r"has not|have not|hasn't|haven't|cannot|can not|can't|never|not) (.+)$", ph)
    if m:
        return ("not", nl_body(m.group(1), term))
    ph = re.sub(r"^(?:is|are|was|were|can|will|has been|have been|be|been) ", "", ph)
    if " who " in ph:
        return nl_restrictor(ph, term)
    return nl_atom(ph, term)

def nl_restrictor(np, term):
    """'people who regularly drink coffee' -> drink_coffee-style conjunction about term."""
    parts = re.split(r"\s+(?:who|that|which)\s+", np, maxsplit=1)
    head = parts[0].strip()
    forms = []
    head = " ".join(w for w in head.split() if w not in GENERIC)
    if head and nl_pred(head):
        forms.append(nl_atom(head, term))
    if len(parts) == 2:
        forms.append(nl_body(parts[1], term))
    if not forms:
        return ("pred", "_top", (term,))           # 'a thing' / 'everything': no restriction
    f = forms[0]
    for g in forms[1:]:
        f = ("and", f, g)
    return f

def _is_name_tok(t):
    return t[:1].isupper() or (any(c.isupper() for c in t) and any(c.isdigit() for c in t))

def nl_name_and_rest(orig):
    toks = orig.split()
    start = 0
    if len(toks) > 2 and toks[0] == "The" and _is_name_tok(toks[1]):
        start = 1                                    # "The Emmet Building ..." is a proper name
    elif not toks or toks[0].lower() in QUANT_HEADS or not _is_name_tok(toks[0]):
        return None
    k = start + 1
    while k < len(toks) and (_is_name_tok(toks[k]) or (toks[k] in ("of", "de", "and") and k + 1 < len(toks) and _is_name_tok(toks[k + 1]) and toks[k] != "and")):
        k += 1
    if k >= len(toks):
        return None
    return "_".join(re.sub(r"\W", "", t.lower()) for t in toks[start:k]), " ".join(toks[k:]).lower()

def nl_clause(text, var):
    """Clause in an if-then: subject is a variable word, a pronoun or a name."""
    low = text.strip().lower()
    m = re.match(r"^(?:someone|something|a person|a thing|anyone|anything|they|he|she|it|that person|the person|x) (.+)$", low)
    if m:
        return nl_body(m.group(1), ("var", var))
    nr = nl_name_and_rest(text.strip())
    if nr:
        return nl_body(nr[1], ("const", nr[0]))
    raise FOLError("clause")

LEAD_INS = [
    (r"^(?:everyone|everybody|anyone|anybody|all people|people) (who|that) ", r"all people \1 "),
    (r"^(?:everything|anything) (that|which) ", r"all things \1 "),
    (r"^(?:everyone|everybody|everything|people|a person|a thing|someone|something) (?=\w)", "all things "),
    (r"^(?:a|an) (?!person\b|thing\b)([\w-]+(?: [\w-]+)?) (?=is |are |was |has |can |will |does |do |either )", r"every \1 "),
    (r"^nobody|^no one", "no person"),
    (r"\b(?:will|would|may|might|must|should|could) (?:be )?", ""),
]
VERBISH = re.compile(r"^(?:is|are|was|were|has|have|had|can|do|does|did|not|never|either|neither|"
                     r"[a-z]+(?:s|ed))$")

STRONG_VERB = re.compile(r"^(?:is|are|was|were|has|have|had|can|do|does|did|not|never|either|neither)$")

def _split_np_vp(rest):
    """'animals breathe' / 'citizens of lawton park use the zip code' -> (NP, VP) by a light verb heuristic."""
    w = rest.split()
    if len(w) < 2:
        return None
    for i in range(1, len(w)):
        if w[i] in ("who", "that", "which"):
            # relative clause: its verb is next; the main verb is the next verb-like token after that
            for pattern in (STRONG_VERB, VERBISH):
                for j in range(i + 2, len(w)):
                    if pattern.match(w[j]) and w[j - 1] not in ("who", "that", "which", "not", "to"):
                        return " ".join(w[:j]), " ".join(w[j:])
            return None
        if w[i] in ("of", "in", "from", "with", "at", "by", "for", "on"):
            continue
        if i == 1 or VERBISH.match(w[i]):
            if i == 1 and not VERBISH.match(w[i]) and len(w) > 2 and VERBISH.match(w[2]) and w[1] not in ("of",):
                continue                              # 'young children are' -> NP of two words
            return " ".join(w[:i]), " ".join(w[i:])
    return None

_IF_STATE = {}

def nl_clause_any(text, conseq=False):
    t = text.strip()
    t = re.sub(r"^(?:all things|someone|something|a person|a thing|anyone|anything|they|he|she|it|that person|the person|x) ", "", t) if re.match(r"^(?:all things|someone|something|a person|a thing|anyone|anything|they|he|she|it|that person|the person|x) ", t) else t
    m = re.match(r"^(?:a|an|the) ([\w-]+(?: [\w-]+)?) ((?:is|are|was|were|has|have|can|does|do|either|neither|not)\b.*|[a-z]+s\b.*)$", t)
    if m and not conseq:
        _IF_STATE["noun"] = m.group(1)
        return ("and", nl_atom(m.group(1), ("var", "x")), nl_body(m.group(2), ("var", "x")))
    if m and conseq and m.group(1) == _IF_STATE.get("noun"):
        return nl_body(m.group(2), ("var", "x"))
    return nl_body(t, ("var", "x"))

def nl_to_fol(sentence):
    orig = sentence.strip().rstrip(".").strip()
    low = re.sub(r"\s+", " ", orig.lower())
    low = low.replace("doesn't", "does not").replace("don't", "do not").replace("isn't", "is not").replace("aren't", "are not")
    for pat, rep in LEAD_INS:
        low = re.sub(pat, rep, low)
    low = re.sub(r"^when ", "if ", low)
    x = ("var", "x")
    m = re.match(r"^(?:all|every|each|any) (.+? (?:who|that|which) (?:is|are|was|were) .+?) (?:is|are) (.+)$", low) \
        or re.match(r"^(?:all|every|each|any) (.+?) (?:is|are) (.+)$", low)
    if m and re.search(r"\b(?:who|that|which)$", m.group(1)):
        m = None
    if m:
        return ("all", "x", ("imp", nl_restrictor(m.group(1), x), nl_body(m.group(2), x)))
    m = re.match(r"^(?:all|every|each|any) (.+?) (?:who|that) (.+?) (\w+s?) (.+)$", low)
    if m and m.group(3) not in DROP:
        return ("all", "x", ("imp", nl_restrictor(m.group(1) + " who " + m.group(2), x),
                             nl_body(m.group(3) + " " + m.group(4), x)))
    m = re.match(r"^no (.+?) (?:is|are|can|will|has|have) (.+)$", low)
    if m:
        return ("all", "x", ("imp", nl_restrictor(m.group(1), x), ("not", nl_body(m.group(2), x))))
    m = re.match(r"^(?:some|there (?:is|are|exists?) (?:a|an|some|at least one)) (.+?) (?:is|are|who|that) (.+)$", low)
    if m:
        return ("ex", "x", ("and", nl_restrictor(m.group(1), x), nl_body(m.group(2), x)))
    m = re.match(r"^(all|every|each|any|some|no) (.+)$", low)
    if m:
        sp = _split_np_vp(m.group(2))
        if sp:
            r_, b_ = nl_restrictor(sp[0], x), nl_body(sp[1], x)
            if m.group(1) == "some":
                return ("ex", "x", ("and", r_, b_))
            if m.group(1) == "no":
                return ("all", "x", ("imp", r_, ("not", b_)))
            return ("all", "x", ("imp", r_, b_))
    m = re.match(r"^if (.+?),? then (.+)$", low) or re.match(r"^if (.+?), (.+)$", low)
    if m:
        a, b = nl_clause_any(m.group(1)), nl_clause_any(m.group(2), conseq=True)
        return ("all", "x", ("imp", a, b))
    nr = nl_name_and_rest(orig)
    if nr:
        name, rest = nr
        rest = rest.replace("doesn't", "does not").replace("isn't", "is not")
        return nl_body(rest, ("const", name))
    raise FOLError("no pattern")

def solve_folio_nl(premises_text, conclusion_text, return_info=False):
    sents = premises_text if isinstance(premises_text, list) else split_sentences(premises_text)
    forms, failed = [], 0
    for s in sents:
        try:
            forms.append(nl_to_fol(s))
        except (FOLError, RecursionError, IndexError):
            failed += 1
    try:
        concl = nl_to_fol(conclusion_text)
        proved, steps = fol_entails(forms, concl)
    except (FOLError, RecursionError, IndexError):
        proved, steps = CONFIG["DEFAULT_LABEL"], 0
        concl = None
    if return_info:
        return proved, {"parsed": len(forms), "failed": failed, "conclusion_parsed": concl is not None}
    return proved

def solve_folio_gold(premises_fol, conclusion_fol, strategy=None):
    """Oracle experiment: use the dataset's annotated FOL instead of our parser."""
    prem = premises_fol if isinstance(premises_fol, list) else [l for l in premises_fol.split("\n") if l.strip()]
    try:
        forms = [fol_parse(p) for p in prem]
        return fol_entails(forms, fol_parse(conclusion_fol), strategy=strategy)[0]
    except (FOLError, RecursionError, IndexError):
        return CONFIG["DEFAULT_LABEL"]

## 7. Router, data loading, evaluation, submission

In [8]:
# ============================================================
# 7. Router: one entry point for any (context, statement) pair
# ============================================================
RT_CACHE = {}          # context -> saturated ForwardChainer (closure computed once per KB)

def looks_counterbench(text):
    t = text.lower()
    return bool(re.search(r"\bwould \S.*? (occur|happen)\b", t)) and ("cause" in t or "direct effect" in t)

def rt_kb_for(context, extra_entities=()):
    """Parse a RuleTaker-style KB once and build the chosen engine; None if not rule-base English.
    extra_entities: individuals named only in the question; rules about 'something' apply to them too."""
    key = (context, tuple(sorted(extra_entities))) if extra_entities else context
    if key in RT_CACHE:
        return RT_CACHE[key]
    facts, rules, _, unparsed = parse_ruletaker(context)
    n = len(split_sentences(context))
    kb = None
    if n and len(unparsed) <= 0.25 * n and (facts or rules):
        if CONFIG["RT_ENGINE"] == "bc":
            kb = BackwardChainer(facts, rules, CONFIG["RULETAKER_MODE"])
            kb.entities |= set(extra_entities)
        else:
            kb = ForwardChainer(facts, rules, CONFIG["RULETAKER_MODE"])
            kb.entities |= set(extra_entities)
            kb.saturate()
        kb.unparsed = unparsed
    RT_CACHE[key] = kb
    return kb

def rt_query(statement):
    q = parse_conj(norm(statement))
    if q is None or any(VAR in a[1:] or PRON in a[1:] for a, _ in q):
        return None
    return q

def predict(context, statement, premises_fol=None, conclusion_fol=None):
    """Returns (label: bool, route: str)."""
    try:
        if premises_fol and conclusion_fol:          # input already supplies a formal representation
            return solve_folio_gold(premises_fol, conclusion_fol), "fol-annotated"
        if isinstance(context, list):
            context = " ".join(str(c).strip().rstrip(".") + "." for c in context)
        if looks_counterbench(context + " " + statement):
            return solve_counterbench(context + " " + statement), "counterbench"
        kb = rt_kb_for(context)
        q = rt_query(statement) if kb is not None else None
        if kb is not None and q is not None:
            missing = {t for a, _ in q for t in a[1:]} - kb.entities
            if missing:                               # question names someone the context never mentions
                kb = rt_kb_for(context, missing)
            return all(kb.query(l) for l in q), "ruletaker"
        if CONFIG.get("TRANSLATOR", "off") == "trained":          # Method 2
            return solve_folio_trained(context, statement), "fol-trained"
        return solve_folio_nl(context, statement), "fol"
    except Exception:
        return CONFIG["DEFAULT_LABEL"], "error"

# ============================================================
# 8. Dataset loading (Hugging Face).  Everything is wrapped so a missing
#    dataset (e.g. gated FOLIO without login) does not stop the notebook.
# ============================================================
def _hf(name, *args):
    from datasets import load_dataset
    return load_dataset(name, *args)

def load_ruletaker():
    ds = _hf("tasksource/ruletaker")
    out = {}
    for split in ds:
        out[split] = [{"id": f"ruletaker-{split}-{i}", "source": "ruletaker", "context": r["context"],
                       "statement": r["question"], "label": to_bool_label(r["label"]),
                       "group": r.get("config", "")} for i, r in enumerate(ds[split])]
    return out

CB_SKIP = ("reason", "explan", "rationale", "solution", "cot", "step", "graph", "id", "type", "level", "meta", "answer", "label", "formal", "rung", "sensical", "variant")
CB_FILES = ["data_balanced_alpha_V1.json", "data_balanced_backdoor_V2.json"]

def _cb_rows(data, tag):
    rows = []
    for i, r in enumerate(data):
        cols = list(r.keys())
        lab = next((c for c in cols if c.lower() in ("answer", "label", "gold", "ground_truth", "target", "gt")), None)
        typ = next((c for c in cols if "type" in c.lower()), None)
        # the 'given' text first, then the question, so the parser sees background before the query
        txt = [c for c in cols if isinstance(r[c], str) and not any(k in c.lower() for k in CB_SKIP)]
        txt.sort(key=lambda c: ("question" in c.lower(), c))
        rows.append({"id": f"counterbench-{tag}-{i}", "source": "counterbench",
                     "context": " ".join(r[c] for c in txt), "statement": "",
                     "label": to_bool_label(r[lab]) if lab else None,
                     "group": f"{tag}:{r[typ]}" if typ else tag})
    return rows

def load_counterbench():
    """The repository mixes files with different columns, so each labelled file is read on its own."""
    from huggingface_hub import hf_hub_download
    out = {}
    for fn in CB_FILES:
        path = hf_hub_download("CounterBench/CounterBench", fn, repo_type="dataset")
        raw = open(path, encoding="utf-8").read()
        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            data = [json.loads(l) for l in raw.splitlines() if l.strip()]
        if isinstance(data, dict):
            data = next((v for v in data.values() if isinstance(v, list)), [data])
        tag = fn.split("_")[2]
        print(f"CounterBench {fn}: {len(data)} rows, columns={list(data[0].keys())}")
        out[tag] = _cb_rows(data, tag)
    return out

FOLIO_GITHUB = "https://raw.githubusercontent.com/Yale-LILY/FOLIO/main/data/v0.0/folio-{}.jsonl"

def load_folio():
    try:
        ds = _hf("yale-nlp/FOLIO")
    except Exception as e:                       # gated on the Hub: fall back to the public GitHub release
        import urllib.request
        print(f"FOLIO on Hugging Face unavailable ({type(e).__name__}); using the public GitHub release v0.0")
        ds = {}
        for split in ["train", "validation"]:
            with urllib.request.urlopen(FOLIO_GITHUB.format(split), timeout=60) as f:
                ds[split] = [json.loads(l) for l in f.read().decode("utf-8").splitlines() if l.strip()]
    out = {}
    for split in ds:
        rows = []
        for i, r in enumerate(ds[split]):
            prem = r["premises"]
            if isinstance(prem, list):
                prem = " ".join(p.strip() if p.strip().endswith(".") else p.strip() + "." for p in prem)
            rows.append({"id": f"folio-{split}-{i}", "source": "folio", "context": prem,
                         "statement": r["conclusion"], "label": to_bool_label(r["label"]),
                         "group": str(r["label"]), "premises_fol": r.get("premises-FOL"),
                         "conclusion_fol": r.get("conclusion-FOL")})
        out[split] = rows
    return out

def safe_load(fn, name):
    try:
        d = fn()
        print(f"Loaded {name}: " + ", ".join(f"{k}={len(v)}" for k, v in d.items()))
        return d
    except Exception as e:
        print(f"Could not load {name}: {type(e).__name__}: {str(e)[:150]}")
        return {}

def pick_split(d, prefs):
    for p in prefs:
        if p in d:
            return d[p]
    return next(iter(d.values()), [])

# ============================================================
# 9. Evaluation helpers
# ============================================================
def evaluate(rows, name, limit=None, verbose=True, use_fol=True):
    rows = rows[:limit] if limit else rows
    RT_CACHE.clear()
    correct, routes, by_group = 0, Counter(), defaultdict(lambda: [0, 0])
    preds = []
    t0 = time.perf_counter()
    for r in rows:
        fol = (r.get("premises_fol"), r.get("conclusion_fol")) if use_fol else (None, None)
        y, route = predict(r["context"], r["statement"], *fol)
        preds.append(y)
        routes[route] += 1
        if r["label"] is not None:
            ok = (y == r["label"])
            correct += ok
            g = by_group[r.get("group", "")]
            g[0] += ok; g[1] += 1
    dt = time.perf_counter() - t0
    acc = 100.0 * correct / max(1, sum(1 for r in rows if r["label"] is not None))
    if verbose:
        print(f"{name}: n={len(rows)}  accuracy={acc:.2f}%  time={dt:.3f}s  "
              f"({1000*dt/max(1,len(rows)):.3f} ms/query)  routes={dict(routes)}")
        if len(by_group) > 1:
            for g, (c, n) in sorted(by_group.items()):
                print(f"    {g:<22} {100*c/n:6.2f}%  (n={n})")
    return {"name": name, "n": len(rows), "acc": acc, "time": dt, "preds": preds}

def write_submission(rows, preds, path="submission.csv", label_col="label"):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", label_col])
        for r, p in zip(rows, preds):
            w.writerow([r["id"], "True" if p else "False"])
    print(f"Wrote {len(rows)} predictions to {path}")

def load_test_file(path):
    """Read a private-test style file (csv / json / jsonl) with id, context and statement columns."""
    low = path.lower()
    if low.endswith((".csv", ".tsv")):
        with open(path, newline="", encoding="utf-8") as f:
            data = list(csv.DictReader(f, delimiter="\t" if low.endswith(".tsv") else ","))
    elif low.endswith(".jsonl"):
        data = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    elif low.endswith(".parquet"):
        import pandas as pd
        data = pd.read_parquet(path).to_dict("records")
    else:
        data = json.load(open(path, encoding="utf-8"))
        if isinstance(data, dict):                       # {"data": [...]} style wrappers
            data = next((v for v in data.values() if isinstance(v, list)), [data])
    def col(row, names):
        for n in names:
            for k in row:
                v = row[k]
                if k.lower() == n and v is not None and not (isinstance(v, float) and v != v) and v != "":
                    return v                      # skip None / NaN / empty cells from mixed tables
        return ""
    rows = []
    for i, r in enumerate(data):
        rows.append({"id": col(r, ["id", "idx", "qid", "example_id", "question_id", "uid"]) or i,
                     "context": col(r, ["context", "premises", "theory", "kb", "knowledge_base", "background", "given_info", "facts", "question"]) or "",
                     "statement": col(r, ["statement", "query", "hypothesis", "conclusion", "assertion", "claim", "question"]) or "",
                     "premises_fol": col(r, ["premises-fol", "premises_fol"]) or None,
                     "conclusion_fol": col(r, ["conclusion-fol", "conclusion_fol"]) or None,
                     "label": None, "source": "test"})
        if rows[-1]["context"] == rows[-1]["statement"]:
            rows[-1]["statement"] = ""
    return rows

## 8. Reasoning traces

In [9]:
# ============================================================
# 10. Reasoning traces (for the report)
# ============================================================
def lit_str(l):
    (a, p) = l
    name = a[0].split(":", 1)[1]
    return ("NOT " if not p else "") + f"{name}({', '.join(a[1:])})"

def trace_ruletaker(context, statement):
    facts, rules, query, unparsed = parse_ruletaker(context, statement)
    kb = ForwardChainer(facts, rules, CONFIG["RULETAKER_MODE"]).saturate()
    print("Facts :", ", ".join(lit_str(f) for f in facts))
    for r in rules:
        print(f"Rule  : {' ∧ '.join(lit_str(p) for p in r.premises)}  →  {lit_str(r.head)}")
    print("Query :", " ∧ ".join(lit_str(q) for q in query) if query else "(unparsed)")
    if query:
        for q in query:
            print("\n".join(kb.explain(q)))
        print("ANSWER:", all(kb.query(q) for q in query), f"  [{kb.inferences} rule firings]")

def trace_counterbench(text):
    ans, q = solve_counterbench(text, return_info=True)
    def show(e):
        if e[0] == "not": return "NOT(" + show(e[1]) + ")"
        if e[0] == "or": return " OR ".join(show(x) for x in e[1]) if len(e[1]) > 1 else show(e[1][0])
        if e[0] == "and": return "(" + " AND ".join(show(x) for x in e[1]) + ")" if len(e[1]) > 1 else show(e[1][0])
        return ("NOT " if e[2] else "") + e[1]
    for v, e in q["eqs"].items():
        print(f"  {v} := {show(e)}")
    print("  observed:", q["observed"], " factual:", q["factual"], " do():", q["do"], " target:", q["target"])
    roots = {v: dict(q["observed"] + q["factual"]).get(v, CONFIG["CB_UNKNOWN_ROOT"]) for v in q["vars"] if v not in q["eqs"]}
    world = scm_eval(q["eqs"], roots, dict(q["do"]), q["vars"])
    print("  counterfactual world:", {k: int(v) for k, v in world.items()})
    print("  question type:", cb_qtype(text), " policy:", CONFIG["CB_POLICY"].get(cb_qtype(text)))
    print("ANSWER:", ans)

def trace_fol(premises, conclusion, gold=False):
    if gold:
        prem = premises if isinstance(premises, list) else [l for l in premises.split("\n") if l.strip()]
        forms, concl = [fol_parse(p) for p in prem], fol_parse(conclusion)
    else:
        sents = premises if isinstance(premises, list) else split_sentences(premises)
        forms = []
        for s in sents:
            try:
                forms.append(nl_to_fol(s)); print("  parsed :", s)
            except FOLError:
                print("  skipped:", s)
        concl = nl_to_fol(conclusion)
    kb = [c for f in forms for c in to_cnf(f)]
    goal = to_cnf(("not", concl))
    print(f"  {len(kb)} KB clauses, negated goal: {[sorted(map(str, c)) for c in goal]}")
    proof = []
    ok, steps = resolution_prove(kb, goal, trace=proof)
    print(f"ANSWER: {ok}  ({steps} given-clause steps)")
    if ok:
        print("  last resolution step between:", [sorted(map(str, c)) for c in proof])

## 8b. Method 2: trained English → FOL translator (code)
Our trained translator only converts English into first-order formulas; the resolution prover still decides every answer. Selected in section 0; evaluated in section 13b.

In [10]:
# ============================================================
# 11. METHOD 2: our trained English -> first-order-logic translator
#     A flan-t5-base model we fine-tuned on FOLIO's training stories (training code: section 17).
#     It ONLY translates English into formulas; every True/False answer is decided by our
#     resolution prover. Outputs are validated (parse + no self-contradiction); anything
#     unusable falls back to Method 1 (rule-based), so the notebook always runs.
# ============================================================
import hashlib

CONFIG.update({"TRANSLATOR": "off"})     # "off" = Method 1 only, "trained" = Method 2 (set in section 13b)
TR_CACHE = {}                            # problem -> translation (so each problem is translated once)
TR_STATS = Counter()                     # what happened to each translation, for reporting

def _premise_list(premises):
    if isinstance(premises, list):
        return [p.strip() for p in premises if str(p).strip()]
    return [p + "." for p in split_sentences(premises)]

def _cache_key(premises, conclusion):
    return hashlib.sha1(json.dumps([premises, conclusion], ensure_ascii=False).encode()).hexdigest()

def trained_translate(premises, conclusion):
    """Return (premise_formulas, conclusion_formula) as parsed ASTs, or None if unusable."""
    prem = _premise_list(premises)
    key = _cache_key(prem, conclusion)
    if key not in TR_CACHE:
        try:
            t5_translate_batch([(prem, conclusion)])
        except Exception as e:
            TR_STATS["model not available: " + type(e).__name__] += 1
            CONFIG["TRANSLATOR"] = "off"           # stop trying for every problem
            return None
    raw = TR_CACHE[key]
    try:
        data = json.loads(raw)
        concl = fol_parse(data["conclusion"])
    except Exception:
        TR_STATS["unusable translation"] += 1
        return None
    forms = []
    for f in data.get("premises", []):
        try:
            forms.append(fol_parse(f))
        except Exception:
            TR_STATS["premise formula dropped"] += 1
    # a self-contradictory translation would 'prove' anything, so reject it
    # (set of support searches from the goal, so here the premises themselves are the support set)
    try:
        kb = [c for f in forms for c in to_cnf(f)]
        if resolution_prove([], kb, max_steps=60)[0]:
            TR_STATS["inconsistent translation rejected"] += 1
            return None
    except Exception:
        return None
    TR_STATS["translated"] += 1
    return forms, concl

def solve_folio_trained(premises, conclusion):
    """Method 2: our model translates, our prover decides; falls back to Method 1."""
    tr = trained_translate(premises, conclusion) if CONFIG.get("T5_MODEL") else None
    if tr is None:
        TR_STATS["fallback to Method 1"] += 1
        return solve_folio_nl(premises, conclusion)
    try:
        return fol_entails(tr[0], tr[1])[0]
    except Exception:
        return CONFIG["DEFAULT_LABEL"]

def needs_translation(r, ignore_fol=False):
    """True only for rows that will reach the English->FOL path (never RuleTaker / CounterBench)."""
    ctx = r["context"] if isinstance(r["context"], str) else " ".join(map(str, r["context"]))
    if r.get("premises_fol") and r.get("conclusion_fol") and not ignore_fol:
        return False
    if looks_counterbench(ctx + " " + r["statement"]):
        return False
    return rt_kb_for(ctx) is None or rt_query(r["statement"]) is None

def translate_prefetch(rows, ignore_fol=False):
    """Translate all problems that need it in batches before the timed prediction loop.
    ignore_fol=True: also translate rows that carry annotated formulas (English-only evaluation)."""
    if CONFIG["TRANSLATOR"] == "off" or not CONFIG.get("T5_MODEL"):
        return
    todo = []
    for r in rows:
        if needs_translation(r, ignore_fol):
            prem = _premise_list(r["context"])
            if _cache_key(prem, r["statement"]) not in TR_CACHE:
                todo.append((prem, r["statement"]))
    if not todo:                                      # nothing to translate: never load torch or the model
        return
    try:
        t5_translate_batch(todo)
    except Exception as e:                            # no torch / transformers / model: Method 1 for everything
        TR_STATS["model not available: " + type(e).__name__] += 1
        CONFIG["TRANSLATOR"] = "off"

# ------------------------------------------------------------
# Our own trained translator (seq2seq, e.g. flan-t5) - runs on CPU at prediction time.
# Shared text format: used by the training code (section 17) AND by prediction, so both always agree.
# ------------------------------------------------------------
CONFIG.update({"T5_MODEL": None,          # set from TRAINED_MODEL in section 0
               "T5_BATCH": 8, "T5_MAX_NEW_TOKENS": 512,
               "T5_BEAMS": 1,             # beam width for generation
               "T5_NBEST": 1,             # candidates kept; the first one that passes every check is used
               "T5_DEVICE": "auto",       # "auto": GPU if available, else CPU
               "T5_QUANTIZE": False})     # int8 dynamic quantization for CPU inference (faster)
T5_PREFIX = "translate to first-order logic: "
_SYM2WORD = [("∀", " FORALL "), ("∃", " EXISTS "), ("¬", " NOT "), ("∧", " AND "), ("∨", " OR "),
             ("→", " IMPLIES "), ("↔", " IFF "), ("⊕", " XOR ")]

def fol_to_words(f):
    """Spell logic symbols as words: T5's vocabulary has no ∀, ∃, ⊕ ..."""
    f = f.replace("⟷", "↔")
    for sym, w in _SYM2WORD:
        f = f.replace(sym, w)
    return re.sub(r"\s+", " ", f).strip()

def words_to_fol(s):
    for sym, w in _SYM2WORD:
        s = re.sub(r"\b" + w.strip() + r"\b", sym, s)
    return re.sub(r"\s+", " ", s).replace("¬ ", "¬").strip()

def t5_source(premises, conclusion):
    sents = list(premises) + [conclusion]
    return T5_PREFIX + " | ".join(f"{i+1}. {s.strip()}" for i, s in enumerate(sents))

def t5_target(formulas):
    return " | ".join(f"{i+1}. {fol_to_words(f)}" for i, f in enumerate(formulas))

def t5_parse_output(text, n):
    """'1. f1 | 2. f2 | ...' -> n formula strings in logic symbols, or None if the count is wrong."""
    parts = [re.sub(r"^\s*\d+\s*\.\s*", "", p) for p in text.split(" | ")]
    parts = [words_to_fol(p) for p in parts if p.strip()]
    return parts if len(parts) == n else None

_T5 = {}
def _t5_reachable(name):
    """5-second check that the model can be downloaded (avoids ~50 s of retries when offline)."""
    import urllib.request
    try:
        urllib.request.urlopen(f"https://huggingface.co/{name}/resolve/main/config.json", timeout=5)
        return True
    except Exception:
        return False

def _t5_load():
    if "model" not in _T5:
        import torch
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        name = CONFIG["T5_MODEL"]
        try:                                           # 1) a local folder, or a copy already downloaded earlier
            _T5["tok"] = AutoTokenizer.from_pretrained(name, local_files_only=True)
            model = AutoModelForSeq2SeqLM.from_pretrained(name, local_files_only=True).eval()
        except Exception:
            if os.path.isdir(name) or not _t5_reachable(name):
                raise OSError("trained translator not available offline")   # 2) no network: fall back at once
            _T5["tok"] = AutoTokenizer.from_pretrained(name)                # 3) download once
            model = AutoModelForSeq2SeqLM.from_pretrained(name).eval()
        if CONFIG["T5_QUANTIZE"] and CONFIG["T5_DEVICE"] == "cpu":
            try:
                model = torch.ao.quantization.quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8)
                TR_STATS["t5 quantized to int8"] += 1
            except Exception as e:
                TR_STATS["t5 quantization unavailable: " + type(e).__name__] += 1
        _T5["model"] = model
        _T5["torch"] = torch
    return _T5

def _translation_ok(fs):
    """A candidate is usable if every formula parses and the premises are not self-contradictory."""
    try:
        forms = [fol_parse(f) for f in fs[:-1]]
        fol_parse(fs[-1])
        return not resolution_prove([], [c for f in forms for c in to_cnf(f)], max_steps=60)[0]
    except Exception:
        return False

def t5_translate_batch(pairs):
    """Translate [(premise_list, conclusion), ...] in batches; keeps the first of the n-best candidates
    that passes every check (falls back to the top candidate). Fills TR_CACHE in the JSON format."""
    t5 = _t5_load()
    tok, model, torch = t5["tok"], t5["model"], t5["torch"]
    if CONFIG["T5_DEVICE"] == "auto":
        CONFIG["T5_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(CONFIG["T5_DEVICE"])
    nbest = max(1, CONFIG["T5_NBEST"])
    beams = max(nbest, CONFIG["T5_BEAMS"])
    for i in range(0, len(pairs), CONFIG["T5_BATCH"]):
        chunk = pairs[i:i + CONFIG["T5_BATCH"]]
        enc = tok([t5_source(p, c) for p, c in chunk], return_tensors="pt", padding=True,
                  truncation=True, max_length=512).to(CONFIG["T5_DEVICE"])
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=CONFIG["T5_MAX_NEW_TOKENS"], num_beams=beams,
                                 num_return_sequences=nbest)
        texts = tok.batch_decode(out, skip_special_tokens=True)
        for k, (p, c) in enumerate(chunk):
            cands = [t5_parse_output(t, len(p) + 1) for t in texts[k * nbest:(k + 1) * nbest]]
            cands = [fs for fs in cands if fs]
            best = next((fs for fs in cands if _translation_ok(fs)), cands[0] if cands else None)
            if best and best is not cands[0]:
                TR_STATS["t5 used a lower-ranked candidate"] += 1
            TR_CACHE[_cache_key(p, c)] = (json.dumps({"premises": best[:-1], "conclusion": best[-1]}, ensure_ascii=False)
                                           if best else "unusable")
            TR_STATS["t5 translations"] += 1

## 9. Offline smoke test (hand-checked examples, runs without any download)

In [11]:
PRE = "Imagine a self-contained, hypothetical world with only the following conditions, and without any unmentioned factors or causal relationships: "
SMOKE = [
 ("Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful.", "Bob is successful.", True),
 ("Anne is nice. Dave is young. Fiona is red. Gary is big. Rough, red people are big. If someone is big and blue then they are white. White people are big. Young, white people are blue. Nice people are big.", "Dave is not blue.", True),
 ("Charlie is not cold. Harry is nice. Harry is white. If Harry is green then Harry is cold. All round things are nice. If something is white and not smart then it is cold.", "Charlie is cold.", False),
 ("The mouse is nice. If something is not green then it is blue. If something is nice and not blue then it is round.", "The mouse is round.", False),
 (PRE + "We know that Nuv causes Splee, Blen and Druk, not Druk causes Plog, Plog causes Skrim, Skrim or Druk causes Zimb, Zimb causes Yurd, and Yurd causes Wrox. Would Wrox occur if not Nuv and not Splee?", "", True),
 (PRE + "We know that Ziklo causes not Blaf, Blaf causes Trune, Trune causes not Vork, Vork or Trune causes Sline, Sline causes Frim, not Frim and Trune causes Qado, and Qado causes Lumbo. Would Lumbo occur if not Ziklo instead of Ziklo?", "", False),
 ("All students are humans. Every human who exercises is healthy. No healthy person is sick. Mary is a student. Mary exercises.", "Mary is not sick.", True),
 ("All students are humans. Every human who exercises is healthy. Mary is a student.", "Mary is healthy.", False),
]
smoke_rows = [{"id": i, "context": c, "statement": s, "label": y, "group": ""} for i, (c, s, y) in enumerate(SMOKE)]
_ = evaluate(smoke_rows, "smoke test")

smoke test: n=8  accuracy=100.00%  time=0.010s  (1.301 ms/query)  routes={'ruletaker': 4, 'counterbench': 2, 'fol': 2}


## 10. Load the public datasets (or switch to prediction-only mode)
If a test file is found next to the notebook (see section 16), the notebook skips all downloads and experiments and only predicts it. Otherwise the public datasets are loaded; FOLIO falls back to its public GitHub release if the Hugging Face copy is gated. Set `EVAL_LIMIT` to a number for a quick run.

In [12]:
EVAL_LIMIT = None      # e.g. 20000 for a quick run; None = full splits
TEST_FILE = None   # e.g. "private_test.csv"; if left as None, a test file next to the notebook is found automatically

def find_test_file():
    """Explicit setting, then the TEST_FILE environment variable, then a file named like a test set in this folder."""
    if TEST_FILE:
        return TEST_FILE
    if os.environ.get("TEST_FILE"):
        return os.environ["TEST_FILE"]
    import glob
    exts = (".csv", ".tsv", ".json", ".jsonl", ".parquet")
    found = [f for f in sorted(glob.glob("*")) if f.lower().endswith(exts) and "test" in f.lower()
             and not f.lower().startswith("submission")]
    for prefer in ("private", "hidden"):
        pref = [f for f in found if prefer in f.lower()]
        if pref:
            return pref[0]
    return found[0] if len(found) == 1 else None

test_path = find_test_file()
PREDICT_ONLY = bool(test_path and os.path.exists(test_path))
if PREDICT_ONLY:
    # grading run: no downloads and no experiments; every experiment section below skips itself,
    # and the settings they chose are already the defaults in CONFIG, so predictions are identical
    print(f"Test file found ({test_path}): prediction-only mode, public datasets and experiments skipped.")
    RT, CB, FO = {}, {}, {}
else:
    RT = safe_load(load_ruletaker, "RuleTaker")
    CB = safe_load(load_counterbench, "CounterBench")
    FO = safe_load(load_folio, "FOLIO")

rt_dev  = pick_split(RT, ["validation", "dev"])
rt_test = pick_split(RT, ["test"])
cb_all  = [r for rows in CB.values() for r in rows]  # both labelled CounterBench files; nothing is trained on them
fo_train = pick_split(FO, ["train"])
fo_test  = pick_split(FO, ["validation", "dev", "test"])
for name, rows in [("RuleTaker", rt_test), ("CounterBench", cb_all), ("FOLIO", fo_test)]:
    if rows:
        r = rows[0]
        print(f"--- {name} example ---\n  context  : {r['context'][:300]}\n  statement: {r['statement']}\n  label    : {r['label']}")

README.md:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

data/train-00000-of-00001-52adaa842dd7ed(…): reconstructing file:   0%|          |  0.00B / 28.6MB            

data/train-00000-of-00001-52adaa842dd7ed(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-402c217dbcd0259(…): reconstructing file:   0%|          |  0.00B / 3.72MB            

data/test-00000-of-00001-402c217dbcd0259(…): downloading bytes:           |  0.00B            

data/dev-00000-of-00001-480b21836eb3db96(…): reconstructing file:   0%|          |  0.00B / 1.85MB            

... (7 outputs left out)

data_balanced_backdoor_V2.json:   0%|          | 0.00/117k [00:00<?, ?B/s]

CounterBench data_balanced_backdoor_V2.json: 200 rows, columns=['question', 'given_info', 'answer', 'type', 'question_id', 'meta']
Loaded CounterBench: alpha=1000, backdoor=200


README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

FOLIO on Hugging Face unavailable (DatasetNotFoundError); using the public GitHub release v0.0
Loaded FOLIO: train=1004, validation=204
--- RuleTaker example ---
  context  : Anne is big. Anne is cold. Anne is furry. Anne is green. Anne is red. Anne is rough. Erin is cold. Erin is furry. Erin is green. Erin is round. If something is cold then it is furry. If something is round then it is rough. If something is green and cold then it is furry. All furry things are round. 
  statement: Anne is red.
  label    : True
--- CounterBench example ---
  context  : We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo. Would Lumbo occur if not Ziklo instead of Ziklo?
  statement: 
  label    : False
--- FOLIO example ---
  context  : If people perform in school talent shows often, then they attend and are very engaged with school events. People either perform in school talent shows often or are inactive and disinterested members of their community. If people

## 11. Parser coverage diagnostics
Sentences the RuleTaker parser could not handle. Use this list to add patterns; it is the main lever on accuracy.

In [13]:
unparsed = Counter()
sample = (rt_dev or rt_test)[:20000]
seen_ctx = set()
for r in sample:
    if r["context"] in seen_ctx:
        continue
    seen_ctx.add(r["context"])
    _, _, _, u = parse_ruletaker(r["context"])
    for s in u:
        unparsed[re.sub(r"\b[A-Z][a-z]+\b", "NAME", s)] += 1
print(f"{len(seen_ctx)} distinct contexts, {sum(unparsed.values())} unparsed sentences")
for s, n in unparsed.most_common(25):
    print(f"{n:6d}  {s}")

1902 distinct contexts, 0 unparsed sentences


## 12. Calibration on development data only
Choices are made on dev/train splits, never on the test split. CounterBench has a single split, so its one setting is chosen on it (reported as such).

In [14]:
if rt_dev:
    mode_acc = {}
    for mode in ["CWA", "OWA"]:
        CONFIG["RULETAKER_MODE"] = mode
        mode_acc[mode] = evaluate(rt_dev, f"RuleTaker dev [{mode}]", limit=20000)["acc"]
    best = max(mode_acc, key=mode_acc.get)
    CONFIG["RULETAKER_MODE"] = best
    print("-> RuleTaker mode:", best)
if cb_all:
    scores = {}
    for v in [False, True]:
        CONFIG["CB_UNKNOWN_ROOT"] = v
        scores[v] = evaluate(cb_all, f"CounterBench [unknown root = {v}]")["acc"]
    CONFIG["CB_UNKNOWN_ROOT"] = max(scores, key=scores.get)
    print("-> CounterBench unknown-root default:", CONFIG["CB_UNKNOWN_ROOT"])
if fo_train:
    counts = Counter(r["label"] for r in fo_train)
    CONFIG["DEFAULT_LABEL"] = counts[True] > counts[False]
    print("FOLIO train label counts (True / not True):", dict(counts), "-> fallback label:", CONFIG["DEFAULT_LABEL"])
print(CONFIG)

RuleTaker dev [CWA]: n=20000  accuracy=100.00%  time=2.514s  (0.126 ms/query)  routes={'ruletaker': 20000}
    depth-1                100.00%  (n=8870)
    depth-3                100.00%  (n=1644)
    depth-5                100.00%  (n=9486)
RuleTaker dev [OWA]: n=20000  accuracy=74.78%  time=1.134s  (0.057 ms/query)  routes={'ruletaker': 20000}
    depth-1                 74.66%  (n=8870)
    depth-3                 72.45%  (n=1644)
    depth-5                 75.31%  (n=9486)
-> RuleTaker mode: CWA
CounterBench [unknown root = False]: n=1200  accuracy=94.58%  time=0.258s  (0.215 ms/query)  routes={'counterbench': 1200}
    alpha:basic            100.00%  (n=250)
    alpha:conditional       92.00%  (n=250)
    alpha:joint             96.00%  (n=250)
    alpha:nested            96.00%  (n=250)
    backdoor:backdoor       87.50%  (n=200)
CounterBench [unknown root = True]: n=1200  accuracy=95.00%  time=0.246s  (0.205 ms/query)  routes={'counterbench': 1200}
    alpha:basic            10

## 12b. Error analysis on RuleTaker dev: negation cycles
Some theories have negation cycles at the predicate level (e.g. *blue if not red*, *red if not blue*). Compares the best-effort ordering with the well-founded semantics (alternating fixpoint), split by whether the theory has a cycle, and keeps the better setting. Uses the dev split only.

In [15]:
def diagnose_negation(rows, limit=30000):
    rows = rows[:limit]
    CONFIG["RT_ENGINE"] = "fc"
    acc, examples = {}, {}
    for setting in ["besteffort", "wfs"]:
        CONFIG["NONSTRATIFIED"] = setting
        RT_CACHE.clear()
        stats, wrong = defaultdict(lambda: [0, 0]), []
        for r in rows:
            y, route = predict(r["context"], r["statement"])
            kb = RT_CACHE.get(r["context"])
            kind = "negation-cycle" if (kb is not None and not kb.stratified) else "stratified"
            ok = (y == r["label"])
            stats[kind][0] += ok; stats[kind][1] += 1
            if not ok and len(wrong) < 50:
                wrong.append((r, y, kind))
        total = sum(c for c, n in stats.values()) / max(1, sum(n for c, n in stats.values()))
        acc[setting], examples[setting] = 100 * total, wrong
        print(f"[{setting:10}] overall {100*total:.2f}%   " +
              "   ".join(f"{k}: {100*c/n:.2f}% (n={n})" for k, (c, n) in sorted(stats.items())))
    best = max(acc, key=acc.get)
    CONFIG["NONSTRATIFIED"] = best
    print("-> using", best)
    return examples[best]

if rt_dev:
    dev_errors = diagnose_negation(rt_dev)
    print(f"\nRemaining dev errors by kind: {Counter(k for _, _, k in dev_errors)}")
    for r, y, kind in dev_errors[:3]:
        print("\n" + "-" * 70)
        print(f"[{r.get('group')}] [{kind}] predicted={y} gold={r['label']}")
        print("CONTEXT :", r["context"])
        print("QUESTION:", r["statement"])
        try:
            trace_ruletaker(r["context"], r["statement"])
        except Exception as e:
            print("trace failed:", e)

[besteffort] overall 98.12%   negation-cycle: 90.85% (n=6162)   stratified: 100.00% (n=23838)
[wfs       ] overall 100.00%   negation-cycle: 100.00% (n=6162)   stratified: 100.00% (n=23838)
-> using wfs

Remaining dev errors by kind: Counter()


## 12c. Engine choice on RuleTaker dev: forward vs backward chaining, end to end
Full dev split, timing includes parsing and the per-knowledge-base cache, exactly as in prediction. Forward chaining is kept unless backward chaining is more accurate, because only forward chaining implements the well-founded semantics exactly.

In [16]:
if rt_dev:
    engine_res = {}
    for eng in ["fc", "bc"]:
        CONFIG["RT_ENGINE"] = eng
        engine_res[eng] = evaluate(rt_dev, f"RuleTaker dev [{eng.upper()}] end-to-end", verbose=True)
    # accuracy first; on a tie keep forward chaining: it implements the well-founded semantics exactly,
    # while backward chaining's loop check can answer negation cycles by search order (seen in our stress tests)
    best = "bc" if engine_res["bc"]["acc"] > engine_res["fc"]["acc"] else "fc"
    CONFIG["RT_ENGINE"] = best
    print("-> RuleTaker engine:", best.upper())

RuleTaker dev [FC] end-to-end: n=75872  accuracy=100.00%  time=4.789s  (0.063 ms/query)  routes={'ruletaker': 75872}
    NatLang                100.00%  (n=3899)
    depth-0                100.00%  (n=8718)
    depth-1                100.00%  (n=8870)
    depth-2                100.00%  (n=8906)
    depth-3                100.00%  (n=8896)
    depth-3ext             100.00%  (n=11587)
    depth-3ext-NatLang     100.00%  (n=15510)
    depth-5                100.00%  (n=9486)
RuleTaker dev [BC] end-to-end: n=75872  accuracy=100.00%  time=4.066s  (0.054 ms/query)  routes={'ruletaker': 75872}
    NatLang                100.00%  (n=3899)
    depth-0                100.00%  (n=8718)
    depth-1                100.00%  (n=8870)
    depth-2                100.00%  (n=8906)
    depth-3                100.00%  (n=8896)
    depth-3ext             100.00%  (n=11587)
    depth-3ext-NatLang     100.00%  (n=15510)
    depth-5                100.00%  (n=9486)
-> RuleTaker engine: FC


## 12d. CounterBench conventions, chosen per question type with a held-out check
The generator's conventions are not stated in the paper, so for each question type six combinations are compared: interventions on endogenous variables (*clamp* = standard do(), *ignore*) × evidence use (*abduce* = observations + factual world, *observed* only, *prior* = Bernoulli priors without conditioning). Ties keep standard semantics (clamp/abduce). The choice is made on even-numbered questions and scored on odd-numbered ones, and vice versa, before the final choice uses all questions.

In [17]:
COMBOS = [("clamp", "abduce"), ("clamp", "observed"), ("clamp", "prior"),
          ("ignore", "abduce"), ("ignore", "observed"), ("ignore", "prior")]
CB_TYPES = ["basic", "joint", "nested", "conditional", "backdoor"]

def cb_grid(rows):
    by_type = defaultdict(list)
    for r in rows:
        by_type[cb_qtype(r["context"])].append(r)
    table = {}
    for t, rs in by_type.items():
        table[t] = {c: sum(solve_counterbench(r["context"], policy=c) == r["label"] for r in rs) / len(rs) for c in COMBOS}
    return table, {t: len(rs) for t, rs in by_type.items()}

def choose_policy(table):
    return {t: max(COMBOS, key=lambda c: table[t][c]) for t in table}   # first = standard semantics wins ties

if cb_all:
    halves = [cb_all[0::2], cb_all[1::2]]
    held = []
    for a, b in [(0, 1), (1, 0)]:
        pol = choose_policy(cb_grid(halves[a])[0])
        CONFIG["CB_POLICY"].update(pol)
        held.append(100 * sum(solve_counterbench(r["context"]) == r["label"] for r in halves[b]) / len(halves[b]))
    print(f"Held-out CounterBench accuracy (choose on one half, score on the other): {held[0]:.2f}% / {held[1]:.2f}%  mean {sum(held)/2:.2f}%\n")
    table, counts = cb_grid(cb_all)
    print(f"{'type':<12}{'n':>5}  " + "".join(f"{e+'/'+v:>16}" for e, v in COMBOS))
    for t in CB_TYPES:
        if t in table:
            print(f"{t:<12}{counts[t]:>5}  " + "".join(f"{100*table[t][c]:>15.1f}%" for c in COMBOS))
    CONFIG["CB_POLICY"].update(choose_policy(table))
    print("\nChosen policy:", {t: "/".join(c) for t, c in CONFIG["CB_POLICY"].items()})
    cb_wrong = [r for r in cb_all if solve_counterbench(r["context"]) != r["label"]]
    print(f"\nRemaining CounterBench errors: {len(cb_wrong)}", Counter(cb_qtype(r["context"]) for r in cb_wrong))
    shown = Counter()
    for r in cb_wrong:
        t = cb_qtype(r["context"])
        if shown[t] >= 2:
            continue
        shown[t] += 1
        print("-" * 80); print(t, "| gold:", r["label"]); print(r["context"])
        trace_counterbench(r["context"])

Held-out CounterBench accuracy (choose on one half, score on the other): 95.00% / 95.00%  mean 95.00%

type            n      clamp/abduce  clamp/observed     clamp/prior   ignore/abduce ignore/observed    ignore/prior
basic         250            100.0%          100.0%          100.0%          100.0%          100.0%          100.0%
joint         250             96.0%           96.0%           96.0%           90.0%           90.0%           90.0%
nested        250             96.0%           96.0%           96.0%           76.0%           76.0%           76.0%
conditional   250             92.0%           92.0%           92.0%           92.0%           92.0%           92.0%
backdoor      200             90.0%           87.5%           75.0%           90.0%           87.5%           75.0%

Chosen policy: {'basic': 'clamp/abduce', 'joint': 'clamp/abduce', 'nested': 'clamp/abduce', 'conditional': 'clamp/abduce', 'backdoor': 'clamp/abduce'}

Remaining CounterBench errors: 60 Counter({'cond

## 13. Final evaluation: accuracy and inference time

In [18]:
results = []
for name, rows in [("RuleTaker test", rt_test), ("CounterBench", cb_all), ("FOLIO validation", fo_test)]:
    if rows:
        results.append(evaluate(rows, name, limit=EVAL_LIMIT))
res_m1 = None
if fo_test:
    print("\nFOLIO from English only (no annotated FOL), reported separately:")
    res_m1 = evaluate(fo_test, "FOLIO validation [English only, Method 1: rule-based]", use_fol=False)
if results:
    n = sum(r["n"] for r in results)
    acc = sum(r["acc"] * r["n"] for r in results) / n
    t = sum(r["time"] for r in results)
    print("=" * 70)
    print(f"OVERALL ACCURACY : {acc:.2f}%  over {n} queries")
    print(f"TOTAL INFERENCE TIME : {t:.2f} s  ({1000*t/n:.3f} ms per query)")
    for r in results:
        print(f"   {r['name']:<20} {r['acc']:6.2f}%   {r['time']:8.2f}s")

RuleTaker test: n=151911  accuracy=100.00%  time=10.428s  (0.069 ms/query)  routes={'ruletaker': 151911}
    NatLang                100.00%  (n=7793)
    depth-0                100.00%  (n=17347)
    depth-1                100.00%  (n=17788)
    depth-2                100.00%  (n=17741)
    depth-3                100.00%  (n=17982)
    depth-3ext             100.00%  (n=23222)
    depth-3ext-NatLang     100.00%  (n=31017)
    depth-5                100.00%  (n=19021)
CounterBench: n=1200  accuracy=95.00%  time=0.414s  (0.345 ms/query)  routes={'counterbench': 1200}
    alpha:basic            100.00%  (n=250)
    alpha:conditional       92.00%  (n=250)
    alpha:joint             96.00%  (n=250)
    alpha:nested            96.00%  (n=250)
    backdoor:backdoor       90.00%  (n=200)
FOLIO validation: n=204  accuracy=98.04%  time=1.350s  (6.616 ms/query)  routes={'fol-annotated': 204}
    False                  100.00%  (n=63)
    True                    94.44%  (n=72)
    Uncertain      

## 13b. Method 2: trained translator, evaluated and compared with Method 1
Always evaluated here when the public data is loaded (skipped in prediction-only mode); `ENGLISH_METHOD` in section 0 only decides which method makes the predictions. A flan-t5-base model we fine-tuned on FOLIO's training stories (training code: appendix, section 17) translates free-form English into first-order logic; **every True/False answer is still decided by our resolution prover**. It is used only for problems that reach the English→FOL path (never RuleTaker or CounterBench). If `torch`/`transformers` or the model are unavailable, Method 1 is used automatically.

In [19]:
# (1) EXPERIMENT: evaluate Method 2 on English-only FOLIO whenever the public data is loaded
#     (skipped automatically in prediction-only mode, so it never costs grading time)
CONFIG["T5_MODEL"] = TRAINED_MODEL                         # our flan-t5-base translator (section 0)
if fo_test:
    CONFIG["TRANSLATOR"] = "trained"
    TR_STATS.clear()
    t0 = time.perf_counter()
    translate_prefetch(fo_test, ignore_fol=True)          # English-only evaluation: translate even annotated rows
    t_translate = time.perf_counter() - t0
    res_m2 = evaluate(fo_test, "FOLIO validation [English only, Method 2: trained translator + our prover]", use_fol=False)
    print(f"Translation time: {t_translate:.1f}s   reasoning time: {res_m2['time']:.2f}s")
    print("Translator statistics:", dict(TR_STATS))
    if CONFIG["TRANSLATOR"] == "off":
        print("NOTE: the trained translator could not be loaded (needs torch, transformers and a one-time download),"
              " so the numbers above come from Method 1.")
    elif res_m1:
        print("\n" + "=" * 70)
        print("METHOD COMPARISON  (FOLIO validation, English only, 204 problems)")
        print(f"  Method 1: rule-based translator     {res_m1['acc']:6.2f}%   {res_m1['time']:8.2f} s")
        print(f"  Method 2: trained translator (ours) {res_m2['acc']:6.2f}%   {t_translate + res_m2['time']:8.2f} s")
        print("  (in both, the resolution prover decides every answer)")

# (2) PREDICTIONS (section 16) use the method chosen in section 0
CONFIG["TRANSLATOR"] = "trained" if ENGLISH_METHOD == "trained" else "off"
print(f"\nPredictions in submission.csv will use: "
      f"{'Method 2 (trained translator)' if ENGLISH_METHOD == 'trained' else 'Method 1 (rule-based, fastest)'}")

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.49k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

FOLIO validation [English only, Method 2: trained translator + our prover]: n=204  accuracy=71.08%  time=3.698s  (18.130 ms/query)  routes={'fol-trained': 200, 'ruletaker': 4}
    False                   95.24%  (n=63)
    True                    27.78%  (n=72)
    Uncertain               94.20%  (n=69)
Translation time: 171.2s   reasoning time: 3.70s
Translator statistics: {'t5 translations': 200, 'translated': 181, 'unusable translation': 11, 'fallback to Method 1': 19, 'inconsistent translation rejected': 8, 'premise formula dropped': 26}

METHOD COMPARISON  (FOLIO validation, English only, 204 problems)
  Method 1: rule-based translator      64.71%       0.16 s
  Method 2: trained translator (ours)  71.08%     174.91 s
  (in both, the resolution prover decides every answer)

Predictions in submission.csv will use: Method 1 (rule-based, fastest)


## 14. Experiments for the report
1. Forward vs backward chaining on RuleTaker.  2. FOLIO: our English parser vs gold FOL annotations (oracle parser).  3. Best-first (unit preference) vs breadth-first resolution.

In [20]:
def stratified_sample(rows, per_group=1000, seed=0):
    """Equal number of questions from every subset (depth-0 ... depth-5, NatLang)."""
    groups = defaultdict(list)
    for r in rows:
        groups[r.get("group", "")].append(r)
    rng = random.Random(seed)
    return {g: rng.sample(v, min(per_group, len(v))) for g, v in sorted(groups.items())}

def compare_engines(sample):
    """Parse every context once (not timed), then time only the reasoning engines."""
    mode = CONFIG["RULETAKER_MODE"]
    print(f"{'subset':<20}{'FC acc':>9}{'BC acc':>9}{'FC ms/q':>10}{'BC ms/q':>10}{'FC fired':>10}{'BC fired':>10}")
    tot = defaultdict(float)
    for g, rows in sample.items():
        parsed = {}
        for r in rows:
            if r["context"] not in parsed:
                f, ru, _, _ = parse_ruletaker(r["context"])
                parsed[r["context"]] = (f, ru)
        qs = [(r, rt_query(r["statement"])) for r in rows]
        qs = [(r, q) for r, q in qs if q is not None]
        # forward chaining: one closure per knowledge base, then lookups
        t0 = time.perf_counter(); closures = {}; fc_ok = 0; fc_fired = 0
        for r, q in qs:
            kb = closures.get(r["context"])
            if kb is None:
                kb = ForwardChainer(*parsed[r["context"]], mode).saturate()
                closures[r["context"]] = kb; fc_fired += kb.inferences
            fc_ok += all(kb.query(l) for l in q) == r["label"]
        fc_t = time.perf_counter() - t0
        # backward chaining: a goal-directed proof per question
        t0 = time.perf_counter(); bc_ok = 0; bc_fired = 0
        for r, q in qs:
            bc = BackwardChainer(*parsed[r["context"]], mode)
            bc_ok += all(bc.prove(a, p) for a, p in q) == r["label"]
            bc_fired += bc.inferences
        bc_t = time.perf_counter() - t0
        n = len(qs)
        for k, v in [("n", n), ("fc", fc_ok), ("bc", bc_ok), ("fct", fc_t), ("bct", bc_t), ("fcf", fc_fired), ("bcf", bc_fired)]:
            tot[k] += v
        print(f"{g:<20}{100*fc_ok/n:8.2f}%{100*bc_ok/n:8.2f}%{1000*fc_t/n:10.4f}{1000*bc_t/n:10.4f}{fc_fired/n:10.1f}{bc_fired/n:10.1f}")
    n = tot["n"]
    print(f"{'ALL':<20}{100*tot['fc']/n:8.2f}%{100*tot['bc']/n:8.2f}%{1000*tot['fct']/n:10.4f}{1000*tot['bct']/n:10.4f}{tot['fcf']/n:10.1f}{tot['bcf']/n:10.1f}")
    print("(fired = rule applications per question; parsing excluded from all timings)")

if rt_test:
    compare_engines(stratified_sample(rt_test, per_group=1000))

if fo_test and fo_test[0].get("premises_fol"):
    base = None
    for limit in [400, 200, 100, 50]:
        t0 = time.perf_counter(); preds = []
        for r in fo_test:
            try:
                prem = r["premises_fol"] if isinstance(r["premises_fol"], list) else [l for l in r["premises_fol"].split("\n") if l.strip()]
                preds.append(fol_entails([fol_parse(x) for x in prem], fol_parse(r["conclusion_fol"]), max_steps=limit)[0])
            except Exception:
                preds.append(CONFIG["DEFAULT_LABEL"])
        base = base or preds
        acc = 100 * sum(p == r["label"] for p, r in zip(preds, fo_test)) / len(fo_test)
        print(f"FOLIO resolution step limit {limit:>3}: accuracy={acc:.2f}%  time={time.perf_counter()-t0:.2f}s  "
              f"predictions changed vs 400: {sum(a != b for a, b in zip(preds, base))}")
    for strat in ["best_first", "bfs"]:
        t0 = time.perf_counter()
        acc = sum(solve_folio_gold(r["premises_fol"], r["conclusion_fol"], strategy=strat) == r["label"] for r in fo_test)
        print(f"FOLIO gold-FOL + resolution [{strat}]  accuracy={100*acc/len(fo_test):.2f}%  time={time.perf_counter()-t0:.2f}s")
    cov = [solve_folio_nl(r["context"], r["statement"], return_info=True)[1] for r in fo_test]
    full = sum(c["failed"] == 0 and c["conclusion_parsed"] for c in cov)
    print(f"FOLIO English parser: fully parsed {full}/{len(cov)} problems; "
          f"{sum(c['parsed'] for c in cov)}/{sum(c['parsed'] + c['failed'] for c in cov)} premises parsed")

subset                 FC acc   BC acc   FC ms/q   BC ms/q  FC fired  BC fired
NatLang               100.00%  100.00%    0.0568    0.0192       2.9       1.5
depth-0               100.00%  100.00%    0.1687    0.0156       3.4       0.3
depth-1               100.00%  100.00%    0.1616    0.0195       3.8       0.5
depth-2               100.00%  100.00%    0.2168    0.0233       5.6       0.9
depth-3               100.00%  100.00%    0.7054    0.0414       8.4       1.3
depth-3ext            100.00%  100.00%    0.2937    0.0283       7.9       1.1
depth-3ext-NatLang    100.00%  100.00%    0.1666    0.0235       7.7       1.3
depth-5               100.00%  100.00%    0.2406    0.0265      12.9       2.3
ALL                   100.00%  100.00%    0.2513    0.0246       6.6       1.2
(fired = rule applications per question; parsing excluded from all timings)
FOLIO resolution step limit 400: accuracy=98.04%  time=17.09s  predictions changed vs 400: 0
FOLIO resolution step limit 200: accuracy

## 15. Example reasoning traces

In [21]:
print("=== RuleTaker (forward chaining with negation as failure) ===")
trace_ruletaker(SMOKE[0][0], SMOKE[0][1]); print()
trace_ruletaker(SMOKE[2][0], "Harry is cold.")
print("\n=== CounterBench (abduction, do-intervention, propagation) ===")
trace_counterbench(SMOKE[4][0])
print("\n=== FOL resolution refutation ===")
trace_fol(SMOKE[6][0], SMOKE[6][1])
if rt_test:
    print("\n=== A deeper RuleTaker example from the test split ===")
    deep = next((r for r in rt_test if "5" in str(r.get("group")) and r["label"]), rt_test[0])
    trace_ruletaker(deep["context"], deep["statement"])

=== RuleTaker (forward chaining with negation as failure) ===
Facts : young(bob), kind(bob)
Rule  : young(?x)  →  smart(?x)
Rule  : smart(?x) ∧ kind(?x)  →  successful(?x)
Query : successful(bob)
successful(bob)   <= rule: "If someone is smart and kind, then they are successful"
    smart(bob)   <= rule: "If someone is young, then they are smart"
        young(bob)   (given fact)
    kind(bob)   (given fact)
ANSWER: True   [3 rule firings]

Facts : NOT cold(charlie), nice(harry), white(harry)
Rule  : green(harry)  →  cold(harry)
Rule  : round(?x)  →  nice(?x)
Rule  : white(?x) ∧ NOT smart(?x)  →  cold(?x)
Query : cold(harry)
cold(harry)   <= rule: "If something is white and not smart then it is cold"
    white(harry)   (given fact)
    NOT smart(harry)   (negation as failure: not derivable)
ANSWER: True   [1 rule firings]

=== CounterBench (abduction, do-intervention, propagation) ===
  splee := nuv
  blen := nuv
  druk := nuv
  plog := NOT druk
  skrim := plog
  zimb := skrim OR druk


## 16. Write `submission.csv`
A test file placed next to the notebook (any csv / tsv / json / jsonl / parquet file with "test" in its name, with `id`, `context` and `statement` columns) is found and predicted automatically; `TEST_FILE` or the `TEST_FILE` environment variable can name it explicitly. Without a test file, predictions for the public test splits are written.

In [22]:
if PREDICT_ONLY:
    test_rows = load_test_file(test_path)
    print(f"Predicting the test file: {test_path} ({len(test_rows)} rows)")
else:
    test_rows = (rt_test[:EVAL_LIMIT] if EVAL_LIMIT else rt_test) + cb_all + fo_test
    print("No test file found: predicting the public test splits instead.")
if not test_rows:
    test_rows = smoke_rows
RT_CACHE.clear()
t0 = time.perf_counter()
if CONFIG["TRANSLATOR"] == "trained":
    translate_prefetch(test_rows)            # Method 2: only rows that need English->FOL translation
preds = [predict(r["context"], r["statement"], r.get("premises_fol"), r.get("conclusion_fol"))[0] for r in test_rows]
print(f"Predicted {len(preds)} queries in {time.perf_counter()-t0:.2f}s")
write_submission(test_rows, preds, "submission.csv")

No test file found: predicting the public test splits instead.
Predicted 153315 queries in 12.35s
Wrote 153315 predictions to submission.csv


## 17. Appendix: how Method 2 was trained (off by default)
The complete training code for our translator. It is **not run** during normal use or grading (`RUN_TRAINING = False`); the prediction sections use the already trained model named in section 0. Set `RUN_TRAINING = True` on a GPU machine to reproduce the model.

In [23]:
# ============================================================
# 17. APPENDIX: how Method 2 (our trained translator) was trained
#     Not run by default (RUN_TRAINING = False): grading only needs the trained model.
#     Set RUN_TRAINING = True on a machine with a GPU to retrain it (~7 min on one RTX A6000):
#       pip install torch transformers accelerate sentencepiece datasets
#     Data: FOLIO v0.0 TRAIN split only. 10% of the training stories are held out for model selection;
#     the validation split is used once, at the end, to report the score.
# ============================================================
RUN_TRAINING = False
TRAIN_SETTINGS = {"base_model": "google/flan-t5-base", "epochs": 15, "lr": 3e-4, "batch": 8,
                  "out": "folio-fol-t5", "push_to_hub": None}     # e.g. "your-hf-name/folio-fol-t5"

def train_method2(base_model, epochs, lr, batch, out, push_to_hub=None, smoke=False):
    import urllib.request, torch
    from datasets import Dataset
    from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                              Seq2SeqTrainer, Seq2SeqTrainingArguments)
    url = "https://raw.githubusercontent.com/Yale-LILY/FOLIO/main/data/v0.0/folio-{}.jsonl"
    fetch = lambda s: [json.loads(l) for l in urllib.request.urlopen(url.format(s), timeout=60).read().decode().splitlines() if l.strip()]
    train_all, valid = fetch("train"), fetch("validation")

    # 1) one entry per story whose annotated formulas our parser accepts
    stories = {}
    for r in train_all:
        prem, fol = [p.strip() for p in r["premises"]], r["premises-FOL"]
        if len(prem) != len(fol) or not prem:
            continue
        try:
            [fol_parse(f) for f in fol]
        except Exception:
            continue
        stories.setdefault(r.get("story_id", len(stories)), (prem, fol))
    ids = sorted(stories); random.Random(0).shuffle(ids)
    dev_ids, tr_ids = set(ids[:max(1, len(ids) // 10)]), ids[max(1, len(ids) // 10):]

    # 2) examples: the last sentence plays the conclusion; each premise in turn is moved to the end
    def examples(story_ids, augment):
        out_ex = []
        for sid in story_ids:
            prem, fol = stories[sid]
            orders = [list(range(len(prem)))]
            if augment:
                orders += [[j for j in range(len(prem)) if j != i] + [i] for i in range(len(prem) - 1)]
            for o in orders:
                P, F = [prem[j] for j in o], [fol[j] for j in o]
                out_ex.append({"src": t5_source(P[:-1], P[-1]), "tgt": t5_target(F)})
        return out_ex
    train_ex, dev_ex = examples(tr_ids, True), examples(sorted(dev_ids), False)
    print(f"stories: {len(tr_ids)} train / {len(dev_ids)} held out -> {len(train_ex)} training examples")

    # 3) model and tokenizer (formulas are spelled with words: T5's vocabulary has no logic symbols)
    if smoke:                                   # tiny random model: only checks that the code runs
        from transformers import ByT5Tokenizer, T5Config, T5ForConditionalGeneration
        tok = ByT5Tokenizer()
        model = T5ForConditionalGeneration(T5Config(vocab_size=384, d_model=32, d_ff=64, num_layers=1, num_heads=2,
                                                    decoder_start_token_id=0, pad_token_id=0, eos_token_id=1))
        train_ex, dev_ex, valid = train_ex[:16], dev_ex[:8], valid[:8]
    else:
        tok = AutoTokenizer.from_pretrained(base_model)
        model = AutoModelForSeq2SeqLM.from_pretrained(base_model)
    exact = sum(tok.decode(tok(e["tgt"]).input_ids, skip_special_tokens=True).strip() == e["tgt"] for e in train_ex)
    print(f"tokenizer round trip on targets: {exact}/{len(train_ex)} exact")

    def tokenize(b):
        enc = tok(b["src"], max_length=512, truncation=True)
        enc["labels"] = tok(text_target=b["tgt"], max_length=512, truncation=True)["input_ids"]
        return enc
    ds_tr = Dataset.from_list(train_ex).map(tokenize, batched=True, remove_columns=["src", "tgt"])
    ds_dev = Dataset.from_list(dev_ex).map(tokenize, batched=True, remove_columns=["src", "tgt"])

    # 4) fine-tune; keep the checkpoint with the lowest held-out loss
    targs = Seq2SeqTrainingArguments(
        output_dir=out + "-ckpt", per_device_train_batch_size=batch, per_device_eval_batch_size=batch,
        learning_rate=lr, num_train_epochs=(1 if smoke else epochs), max_steps=(2 if smoke else -1),
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=2, load_best_model_at_end=True,
        metric_for_best_model="eval_loss", greater_is_better=False, logging_steps=50, report_to=[],
        bf16=(torch.cuda.is_available() and torch.cuda.is_bf16_supported()), use_cpu=smoke)
    trainer = Seq2SeqTrainer(model=model, args=targs, train_dataset=ds_tr, eval_dataset=ds_dev,
                             data_collator=DataCollatorForSeq2Seq(tok, model=model))
    t0 = time.time(); trainer.train(); print(f"training time: {(time.time() - t0) / 60:.1f} min")
    model = trainer.model; model.save_pretrained(out); tok.save_pretrained(out)

    # 5) report on FOLIO validation through OUR prover (the model never decides an answer)
    CONFIG.update({"TRANSLATOR": "trained", "T5_MODEL": out, "T5_DEVICE": "cpu" if smoke else "auto",
                   "T5_MAX_NEW_TOKENS": 8 if smoke else 512})
    _T5.clear(); TR_CACHE.clear(); TR_STATS.clear()
    rows = [{"context": r["premises"], "statement": r["conclusion"]} for r in valid]
    translate_prefetch(rows, ignore_fol=True)
    acc = sum(solve_folio_trained(r["premises"], r["conclusion"]) == to_bool_label(r["label"]) for r in valid)
    print(f"FOLIO validation, English only, Method 2 + our prover: {100 * acc / len(valid):.2f}%   {dict(TR_STATS)}")
    if push_to_hub and not smoke:
        model.push_to_hub(push_to_hub); tok.push_to_hub(push_to_hub)
        print(f"uploaded: https://huggingface.co/{push_to_hub}  -> set TRAINED_MODEL in section 0")
    return out

if RUN_TRAINING:
    train_method2(**TRAIN_SETTINGS)
else:
    print("Training is off (RUN_TRAINING = False); prediction uses the already trained model from section 0.")

Training is off (RUN_TRAINING = False); prediction uses the already trained model from section 0.
